# Experiments v4: uncertainty-guided refinement with bone-length consistency

This notebook produces every number of the IEEE version of the paper. It runs in five parts, and **every result is saved to your Google Drive** (folder `3DPW_results_v4`) as soon as it is computed. If Colab disconnects, open the notebook again and run all cells: finished pieces are skipped and the run continues where it stopped.

| Part | What it does | Time on a T4 GPU |
|---|---|---|
| A | ResNet-50 MeTRAbs model on every 3DPW test and validation frame (all detections and all 5 test-time crops are kept) | about 1 hour |
| B | Real-object occluders (Pascal VOC): a per-frame study at 10–50% coverage, and occlusion *episodes* of 30 frames inside the videos | about 1 hour |
| C | MuPoTS-3D, a second dataset with several people and many occlusions | about 40 minutes |
| D | A larger released model (EfficientNetV2-M) on the 3DPW test split, to show the refinement works with another model | about 1 hour |
| E | All results: tuning on the validation split, comparison with One-Euro, Savitzky–Golay, Gaussian and SmoothNet smoothing, a real tracker, significance tests, number of crops, sensitivity, speed | CPU is enough |
| F–H | Checks added in the revision: shuffled-uncertainty controls, bone-stage ablation, extended search, tracker identity, gap filling and lag, default MuPoTS camera, GPU speed, more occlusion schedules, example figures | about 1.5 hours |
| I | HMR 2.0, an estimator outside the MeTRAbs family, on 3DPW (needs the SMPL model, see Part I) | about 1.5 hours |

**Data and terms.** 3DPW (non-commercial research, cite the ECCV 2018 paper, do not share), Pascal VOC 2012 (images from Flickr; use must respect Flickr's terms of use), MuPoTS-3D (terms are printed in Part C before anything is downloaded), SmoothNet and the MeTRAbs models (non-commercial research only). Datasets are downloaded into this Colab session only, never to your Drive or computer. HMR 2.0 (4D-Humans, MIT licence) and the SMPL body model (registration at smpl.is.tue.mpg.de, non-commercial) are needed only for Part I. Only results go to your Drive.

**How to run:** Runtime → Change runtime type → T4 GPU. Keep `QUICK_TEST = True` first and run all cells (a few minutes) to check everything works; **those numbers are not for the paper**. Then set `QUICK_TEST = False` and run all cells again.

## Step 1 — Settings

In [ ]:
# ===== Settings: edit these if needed =====
QUICK_TEST          = True    # True = short check only. False = the run for the paper.
RUN_3DPW_MAIN       = True    # Part A
RUN_VOC_STUDY       = True    # Part B
RUN_MUPOTS          = True    # Part C
RUN_SECOND_MODEL    = True    # Part D
RUN_ANALYSIS        = True    # Part E
RUN_REVIEW          = False   # Parts F-H (checks added in the revision; run after Parts A-E)
RUN_HMR2            = False   # Part I (HMR 2.0; needs the SMPL model, see Part I)
VOC_TERMS_AGREED    = False   # Pascal VOC 2012 terms (see Part B)
MUPOTS_TERMS_AGREED = False   # MuPoTS-3D terms (printed in Part C)

DATA_URL      = 'https://virtualhumans.mpi-inf.mpg.de/3DPW/'
MAIN_MODEL    = 'metrabs_rn50_y4'     # ResNet-50 + YOLOv4 (the model of the paper)
SECOND_MODEL  = 'metrabs_eff2m_y4'    # EfficientNetV2-M + YOLOv4 (larger released model)
NUM_AUG       = 5                     # test-time crops per person
MAX_GAP       = 15                    # longest gap (frames) the refinement bridges
ROBUST_STRIDE = 15                    # per-frame occluder study: every 15th test frame
OCC_LEVELS    = [0.1, 0.2, 0.3, 0.4, 0.5]          # fraction of the person box covered by the object
EPISODE_EVERY, EPISODE_LEN, EPISODE_COVER = 150, 30, (0.3, 0.5)   # occlusion episodes in the videos
DRIVE_DIR     = '/content/drive/MyDrive/3DPW_results_v4'
SMOOTHNET_FILE_ID = '106MnTXFLfMlJ2W7Fvw2vAlsUuQFdUe6k'   # released SmoothNet checkpoint (3DPW-SPIN-3D)

## Step 2 — Connect Google Drive and check the GPU
Google asks for permission to let this notebook use your Drive. Click **Allow**. Only result files are written there.

In [ ]:
import os, glob, pickle, zipfile, urllib.request, json, time, subprocess, re, gc, shutil
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = os.path.join(DRIVE_DIR, 'quick' if QUICK_TEST else 'full')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results folder in your Drive:', RESULTS_DIR, '|', len(os.listdir(RESULTS_DIR)), 'files already saved')

import tensorflow as tf
from tqdm.auto import tqdm
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print('TensorFlow', tf.__version__, '| GPU:', gpus if gpus else 'NONE (fine for Part E only)')

def save_pkl(obj, path):
    tmp = path + '.part'
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_json(obj, path):
    with open(path + '.part', 'w') as f:
        json.dump(obj, f, indent=1)
    os.replace(path + '.part', path)

## Step 3 — Helper functions (no need to edit)
Reading the data, errors, the Kalman filter and smoother, the new bone-length step, the comparison smoothers, the tracker, the occluders and the statistics. The last lines run self-tests.

In [ ]:
import os, glob, pickle
import numpy as np

SMPL_JOINTS = ['pelv', 'lhip', 'rhip', 'spi1', 'lkne', 'rkne', 'spi2', 'lank', 'rank', 'spi3', 'ltoe', 'rtoe',
               'neck', 'lcla', 'rcla', 'head', 'lsho', 'rsho', 'lelb', 'relb', 'lwri', 'rwri', 'lhan', 'rhan']
ROOT = 0  # pelvis


def load_sequence(pkl_path):
    """3DPW .pkl files were written with Python 2, so latin1 decoding is needed."""
    with open(pkl_path, 'rb') as f:
        return pickle.load(f, encoding='latin1')


def poses2d_xyc(arr):
    """Return 2D keypoints as (18, 3) = x, y, confidence, whatever the stored layout."""
    a = np.asarray(arr, dtype=np.float64)
    return a.T if a.shape[0] == 3 else a


def to_camera(X_world, cam_pose, inverse=False):
    """World coordinates (J, 3) -> camera coordinates using a 4x4 extrinsic matrix."""
    M = np.linalg.inv(cam_pose) if inverse else cam_pose
    return X_world @ M[:3, :3].T + M[:3, 3]


def project(X_cam, K):
    """Pinhole projection of camera coordinates (J, 3) to pixels (J, 2)."""
    uv = X_cam @ K.T
    return uv[:, :2] / uv[:, 2:3]


def similarity_align(pred, gt):
    """Best scale s, rotation R and translation t so that s R pred + t is closest to gt (Procrustes)."""
    mu_p, mu_g = pred.mean(0), gt.mean(0)
    P, G = pred - mu_p, gt - mu_g
    U, S, Vt = np.linalg.svd(G.T @ P)
    D = np.eye(3)
    D[2, 2] = np.sign(np.linalg.det(U @ Vt))
    R = U @ D @ Vt
    s = np.trace(np.diag(S) @ D) / (P ** 2).sum()
    t = mu_g - s * R @ mu_p
    return s * pred @ R.T + t


def joint_errors(pred_mm, gt_mm):
    """Per-joint errors (mm) after aligning the root joint, and after Procrustes alignment."""
    p = pred_mm - pred_mm[ROOT]
    g = gt_mm - gt_mm[ROOT]
    err = np.linalg.norm(p - g, axis=1)
    err_pa = np.linalg.norm(similarity_align(pred_mm, gt_mm) - gt_mm, axis=1)
    return err, err_pa


def summarize(errs, errs_pa):
    """MPJPE, PA-MPJPE, PCK@50mm and AUC (0-200 mm) over a list of per-joint error arrays."""
    E = np.stack(errs)          # (poses, joints)
    EP = np.stack(errs_pa)
    thresholds = np.arange(0, 201, 5)
    pck_curve = [(E <= th).mean() * 100 for th in thresholds]
    return {'MPJPE': E.mean(), 'PA-MPJPE': EP.mean(),
            'PCK@50mm': (E <= 50).mean() * 100, 'AUC@200mm': float(np.mean(pck_curve))}


def camera_convention_scores(seq, step=50):
    """Median 2D distance (px) between projected 3D ground truth and the stored 2D keypoints,
    for cam_poses used as given (False) and inverted (True). The smaller one is the right convention."""
    K = np.asarray(seq['cam_intrinsics'], dtype=np.float64)
    scores = {}
    for inverse in (False, True):
        d = []
        for a in range(len(seq['jointPositions'])):
            J = np.asarray(seq['jointPositions'][a]).reshape(-1, 24, 3)
            for i in range(0, len(J), step):
                if not seq['campose_valid'][a][i]:
                    continue
                Xc = to_camera(J[i], np.asarray(seq['cam_poses'][i]), inverse)
                if (Xc[:, 2] <= 0.1).any():
                    d.append(1e6)
                    continue
                kp = poses2d_xyc(seq['poses2d'][a][i])
                ok = kp[:, 2] > 0.3
                if ok.sum() < 6:
                    continue
                d.append(np.linalg.norm(project(Xc, K).mean(0) - kp[ok, :2].mean(0)))
        scores[inverse] = float(np.median(d)) if d else 1e6
    return scores


def frame_is_valid(seq, actor, i):
    """Official 3DPW rule: camera pose must be valid and at least six 2D keypoints detected."""
    if not seq['campose_valid'][actor][i]:
        return False
    kp = poses2d_xyc(seq['poses2d'][actor][i])
    return (kp[:, 2] > 0).sum() >= 6


def gt_box(uv, img_w, img_h, margin=0.15):
    """Box [left, top, width, height] around projected ground-truth joints, enlarged by a margin."""
    x0, y0 = uv.min(0)
    x1, y1 = uv.max(0)
    w, h = x1 - x0, y1 - y0
    x0, x1 = x0 - margin * w, x1 + margin * w
    y0, y1 = y0 - margin * h, y1 + margin * h
    return [float(x0), float(y0), float(x1 - x0), float(y1 - y0)]


def match_detections(pred2d, gt2d_list):
    """Greedily assign each ground-truth person to the detection whose 2D joints are closest.
    Returns, for each ground-truth person, the detection index or None (missed)."""
    out = [None] * len(gt2d_list)
    if len(pred2d) == 0:
        return out
    cost = np.array([[np.linalg.norm(p - g, axis=1).mean() for p in pred2d] for g in gt2d_list])
    used = set()
    for flat in np.argsort(cost, axis=None):
        gi, di = np.unravel_index(flat, cost.shape)
        if out[gi] is not None or di in used:
            continue
        g = gt2d_list[gi]
        height = g[:, 1].max() - g[:, 1].min()
        if cost[gi, di] <= max(50.0, 0.25 * height):
            out[gi] = int(di)
            used.add(di)
    return out


import numpy as np

# ---------------------------------------------------------------------------
# Uncertainty from test-time augmentation
# ---------------------------------------------------------------------------
def rootrel_mean_and_sigma(pred_aug, root=0):
    # pred_aug: (A, J, 3) absolute joint positions (mm) from A augmented crops.
    # Returns the mean root-relative pose (J, 3) and the per-joint spread sigma (J,) in mm,
    # i.e. the root-mean-square per-axis deviation of the A root-relative predictions.
    rr = pred_aug - pred_aug[:, root:root + 1]
    mean = rr.mean(0)
    sigma = np.sqrt(((rr - mean) ** 2).sum(-1).mean(0) / 3.0)
    return mean, sigma


# ---------------------------------------------------------------------------
# Constant-velocity Kalman filter / Rauch-Tung-Striebel smoother, run independently
# for every joint coordinate (vectorized over D = joints x 3 dimensions).
# ---------------------------------------------------------------------------
def kalman_track(z, r, q, max_gap=15, smooth=False):
    # z: (T, D) measurements, NaN where missing.  r: (T, D) measurement variances.
    # q: process-noise intensity (mm^2 per frame^3).  Gaps longer than max_gap frames
    # start a new segment.  Returns (T, D) estimates (NaN where no estimate exists).
    T, D = z.shape
    out = np.full((T, D), np.nan)
    have = ~np.isnan(z).any(1)
    idx = np.nonzero(have)[0]
    if len(idx) == 0:
        return out
    # split into segments at long gaps
    segs, start = [], idx[0]
    for a, b in zip(idx[:-1], idx[1:]):
        if b - a - 1 > max_gap:
            segs.append((start, a))
            start = b
    segs.append((start, idx[-1]))
    Q00, Q01, Q11 = q / 3.0, q / 2.0, q
    for s, e in segs:
        n = e - s + 1
        xp = np.zeros((n, 2, D)); Pp = np.zeros((n, 3, D))   # predicted (P00, P01, P11)
        xf = np.zeros((n, 2, D)); Pf = np.zeros((n, 3, D))   # filtered
        p = z[s].copy(); v = np.zeros(D)
        P00 = r[s].copy(); P01 = np.zeros(D); P11 = np.full(D, 1e4)
        for k in range(n):
            t = s + k
            if k > 0:  # predict
                p = p + v
                P00, P01, P11 = P00 + 2 * P01 + P11 + Q00, P01 + P11 + Q01, P11 + Q11
            xp[k, 0], xp[k, 1] = p, v
            Pp[k] = P00, P01, P11
            if have[t] and k > 0:  # update
                S = P00 + r[t]
                K0, K1 = P00 / S, P01 / S
                innov = z[t] - p
                p = p + K0 * innov
                v = v + K1 * innov
                P00, P01, P11 = (1 - K0) * P00, (1 - K0) * P01, P11 - K1 * P01
            xf[k, 0], xf[k, 1] = p, v
            Pf[k] = P00, P01, P11
        if smooth:
            ps, vs = xf[-1, 0].copy(), xf[-1, 1].copy()
            out[e] = ps
            for k in range(n - 2, -1, -1):
                f00, f01, f11 = Pf[k]
                a00, a01, a11 = Pp[k + 1]
                det = a00 * a11 - a01 * a01
                i00, i01, i11 = a11 / det, -a01 / det, a00 / det
                # C = Pf F^T Pp^-1 with F = [[1,1],[0,1]]  ->  Pf F^T = [[f00+f01, f01],[f01+f11, f11]]
                b00, b01, b10, b11 = f00 + f01, f01, f01 + f11, f11
                C00 = b00 * i00 + b01 * i01; C01 = b00 * i01 + b01 * i11
                C10 = b10 * i00 + b11 * i01; C11 = b10 * i01 + b11 * i11
                dp, dv = ps - xp[k + 1, 0], vs - xp[k + 1, 1]
                ps = xf[k, 0] + C00 * dp + C01 * dv
                vs = xf[k, 1] + C10 * dp + C11 * dv
                out[s + k] = ps
        else:
            out[s:e + 1] = xf[:, 0]
    return out


def refine_track(frames, poses, sigmas, mode, q, lam=1.0, smin=5.0, rfix=100.0, max_gap=15, smooth=False):
    # frames: sorted frame indices with a measurement; poses: (n, J, 3) root-relative;
    # sigmas: (n, J).  mode: 'fixed' (constant R = rfix) or 'uncertainty' (R = lam*max(sigma, smin)^2).
    # Returns dict frame -> refined pose (J, 3), including frames inside short gaps.
    if len(frames) == 0:
        return {}
    f0, f1 = frames[0], frames[-1]
    T, J = f1 - f0 + 1, poses.shape[1]
    z = np.full((T, J * 3), np.nan)
    r = np.full((T, J * 3), 1.0)
    for f, P, s in zip(frames, poses, sigmas):
        z[f - f0] = P.reshape(-1)
        if mode == 'fixed':
            r[f - f0] = rfix
        else:
            r[f - f0] = np.repeat(lam * np.maximum(s, smin) ** 2, 3)
    est = kalman_track(z, r, q, max_gap=max_gap, smooth=smooth)
    res = {}
    for k in range(T):
        if not np.isnan(est[k]).any():
            res[f0 + k] = est[k].reshape(J, 3)
    return res


def accel_error(pred_seq, gt_seq):
    # pred_seq, gt_seq: (T, J, 3) for consecutive frames. Mean acceleration error (mm/frame^2).
    if len(pred_seq) < 3:
        return np.zeros(0)
    ap = pred_seq[2:] - 2 * pred_seq[1:-1] + pred_seq[:-2]
    ag = gt_seq[2:] - 2 * gt_seq[1:-1] + gt_seq[:-2]
    return np.linalg.norm(ap - ag, axis=-1).mean(-1)


# ---------------------------------------------------------------------------
# Synthetic occlusion and truncation (robustness study)
# ---------------------------------------------------------------------------
def occluder_rect(box, frac, rng):
    # Rectangle covering `frac` of the box area, random aspect ratio in [0.5, 2], random position
    # inside the box. box = [left, top, width, height]. Returns (x0, y0, x1, y1) or None.
    if frac <= 0:
        return None
    x, y, w, h = box
    area = frac * w * h
    ar = np.exp(rng.uniform(np.log(0.5), np.log(2.0)))
    rw, rh = min(np.sqrt(area * ar), w), min(np.sqrt(area / ar), h)
    if rw * rh < area:  # clipped by the box: stretch the other side
        if rw >= w: rh = min(area / rw, h)
        else: rw = min(area / rh, w)
    x0 = x + rng.uniform(0, w - rw)
    y0 = y + rng.uniform(0, h - rh)
    return (x0, y0, x0 + rw, y0 + rh)


def apply_occluder(img, rect, rng):
    # Fill the rectangle with uniform random noise (random erasing). img: (H, W, 3) uint8.
    out = img.copy()
    if rect is None:
        return out
    H, W = img.shape[:2]
    x0, y0, x1, y1 = [int(round(v)) for v in rect]
    x0, x1 = max(0, x0), min(W, x1)
    y0, y1 = max(0, y0), min(H, y1)
    if x1 > x0 and y1 > y0:
        out[y0:y1, x0:x1] = rng.integers(0, 256, size=(y1 - y0, x1 - x0, 3), dtype=np.uint8)
    return out


def inside_rect(uv, rect):
    if rect is None:
        return np.zeros(len(uv), bool)
    x0, y0, x1, y1 = rect
    return (uv[:, 0] >= x0) & (uv[:, 0] <= x1) & (uv[:, 1] >= y0) & (uv[:, 1] <= y1)


def truncation_cut(uv, frac):
    # Image row at which to cut so that the lowest `frac` of the person's height is removed.
    y_top, y_bot = uv[:, 1].min(), uv[:, 1].max()
    return int(np.floor(y_top + (1.0 - frac) * (y_bot - y_top)))


def clip_box(box, W, H):
    x, y, w, h = box
    x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
    return [float(x0), float(y0), float(x1 - x0), float(y1 - y0)]


import time
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d

# ---------------------------------------------------------------------------
# Skeletons: parent of every joint (-1 = root) and left/right bone pairs (by child joint)
# ---------------------------------------------------------------------------
SMPL_PARENTS = np.array([-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21])
SMPL_SYM = [(1, 2), (4, 5), (7, 8), (10, 11), (13, 14), (16, 17), (18, 19), (20, 21), (22, 23)]
MPI17_NAMES = ['htop', 'neck', 'rsho', 'relb', 'rwri', 'lsho', 'lelb', 'lwri', 'rhip', 'rkne', 'rank',
               'lhip', 'lkne', 'lank', 'pelv', 'spin', 'head']
MPI17_PARENTS = np.array([16, 15, 1, 2, 3, 1, 5, 6, 14, 8, 9, 14, 11, 12, -1, 14, 1])
MPI17_SYM = [(2, 5), (3, 6), (4, 7), (8, 11), (9, 12), (10, 13)]
MPI17_ROOT = 14


def topo_order(parents):
    # Joints ordered so that every parent comes before its children.
    order, frontier = [], [int(np.where(parents < 0)[0][0])]
    while frontier:
        j = frontier.pop(0)
        order.append(j)
        frontier += [int(c) for c in np.where(parents == j)[0]]
    return order


# ---------------------------------------------------------------------------
# Errors with a configurable root joint
# ---------------------------------------------------------------------------
def joint_errors_r(pred_mm, gt_mm, root):
    p = pred_mm - pred_mm[root]
    g = gt_mm - gt_mm[root]
    err = np.linalg.norm(p - g, axis=1)
    err_pa = np.linalg.norm(similarity_align(pred_mm, gt_mm) - gt_mm, axis=1)
    return err, err_pa


def summarize_r(E, EP, pck_thr, auc_max, pck_joints=None):
    # E, EP: (poses, joints) errors in mm.  PCK and AUC over `pck_joints` (default: all).
    Ep = E if pck_joints is None else E[:, pck_joints]
    th = np.arange(0, auc_max + 1, 5)
    return {'MPJPE': float(E.mean()), 'PA-MPJPE': float(EP.mean()),
            'PCK': float((Ep <= pck_thr).mean() * 100), 'AUC': float(np.mean([(Ep <= t).mean() * 100 for t in th]))}


# ---------------------------------------------------------------------------
# Bone-length consistency (new): a person's bone lengths do not change over time.
# Lengths are estimated from the confident (low-sigma) frames of a track and then used to
# correct the uncertain joints, blending by the joint's uncertainty.
# ---------------------------------------------------------------------------
def bone_lengths(P, parents):
    ch = np.where(parents >= 0)[0]
    L = np.full(P.shape[:-1], np.nan)
    L[..., ch] = np.linalg.norm(P[..., ch, :] - P[..., parents[ch], :], axis=-1)
    return L


def bone_weights(S, parents, s0):
    ch = np.where(parents >= 0)[0]
    W = np.zeros(S.shape)
    W[..., ch] = 1.0 / (S[..., ch] ** 2 + S[..., parents[ch]] ** 2 + s0 ** 2)
    return W


def weighted_median(x, w):
    # x, w: (n,) -> weighted median of x
    i = np.argsort(x)
    c = np.cumsum(w[i])
    k = int(np.searchsorted(c, 0.5 * c[-1]))
    return float(x[i][min(k, len(x) - 1)])


def bone_groups(parents, sym):
    # Groups of child joints whose bones share one length (left/right pairs if sym).
    ch = [int(j) for j in np.where(parents >= 0)[0]]
    groups, used = [], set()
    if sym:
        for a, b in sym:
            groups.append([a, b]); used |= {a, b}
    groups += [[j] for j in ch if j not in used]
    return groups


def estimate_bones(poses, sigmas, parents, sym=None, s0=5.0, how='median', online=False):
    # poses (n, J, 3), sigmas (n, J).  Offline: one length per bone, shape (J,), NaN at the root.
    # Online: causal estimate for every frame from that frame and the frames before it, shape (n, J).
    L = bone_lengths(poses, parents)
    W = bone_weights(sigmas, parents, s0)
    n, J = L.shape
    out = np.full((n, J) if online else (J,), np.nan)
    for g in bone_groups(parents, sym):
        if online:
            num = np.cumsum((W[:, g] * L[:, g]).sum(1))
            den = np.cumsum(W[:, g].sum(1))
            for j in g:
                out[:, j] = num / den
        else:
            x, w = L[:, g].reshape(-1), W[:, g].reshape(-1)
            v = weighted_median(x, w) if how == 'median' else float((w * x).sum() / w.sum())
            for j in g:
                out[j] = v
    return out


def apply_bones(X, Lhat, parents, order, S, kappa=1.0, smin=3.0, iters=10):
    # Uncertainty-weighted projection of each pose onto the track's bone lengths.
    # X (n, J, 3) root-relative poses; Lhat (J,) or (n, J) target lengths; S (n, J) joint uncertainties.
    # Each bone constraint |Y_j - Y_p| = L is enforced by moving both ends along the bone, the more
    # uncertain end moving more (inverse mass = max(sigma, smin)^2; the root stays fixed). A few
    # Gauss-Seidel sweeps approximate the weighted least-squares projection. kappa in [0, 1] blends
    # between the input (0) and the projected pose (1).
    n, J = X.shape[:2]
    Lf = np.broadcast_to(Lhat, (n, J)) if np.ndim(Lhat) == 1 else Lhat
    w = np.maximum(S, smin) ** 2
    w[:, parents < 0] = 0.0
    Y = X.copy()
    bones = [(j, parents[j]) for j in order if parents[j] >= 0]
    for _ in range(iters):
        for j, p in bones:
            v = Y[:, j] - Y[:, p]
            ln = np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-6)
            ws = w[:, j:j + 1] + w[:, p:p + 1]
            lam = (ln - Lf[:, j:j + 1]) / np.maximum(ws, 1e-9)
            d = v / ln
            Y[:, j] -= w[:, j:j + 1] * lam * d
            Y[:, p] += w[:, p:p + 1] * lam * d
    return X + kappa * (Y - X)


# ---------------------------------------------------------------------------
# Full refinement: optional Kalman filter / RTS smoother (helpers_v3) + optional bone stage
# cfg = dict(mode='fixed'|'uncertainty'|None, q, lam, smin, rfix, smooth,
#            bones=None | dict(kappa, bsmin, s0, how, sym, order='after'|'before'))
# ---------------------------------------------------------------------------
def _bone_stage(X, S, poses, sigmas, frames, fs, bones, smooth, parents, order, sym_pairs):
    sym = sym_pairs if bones.get('sym') else None
    if smooth:
        Lh = estimate_bones(poses, sigmas, parents, sym, bones.get('s0', 5.0), bones.get('how', 'median'))
    else:  # causal: lengths from measured frames up to each frame (gap frames use the last estimate)
        Lc = estimate_bones(poses, sigmas, parents, sym, bones.get('s0', 5.0), online=True)
        pos = np.searchsorted(frames, fs, side='right') - 1
        Lh = Lc[np.maximum(pos, 0)]
    return apply_bones(X, Lh, parents, order, S, bones.get('kappa', 1.0), bones.get('bsmin', 3.0))


def refine_v4(frames, poses, sigmas, cfg, parents, order, sym_pairs, max_gap=15):
    if len(frames) == 0:
        return {}
    frames = np.asarray(frames)
    bones = cfg.get('bones')
    smooth = cfg.get('smooth', False)
    P = poses
    if bones is not None and bones.get('order', 'after') == 'before':
        P = _bone_stage(poses, sigmas, poses, sigmas, frames, frames, bones, smooth, parents, order, sym_pairs)
    if cfg.get('mode') is None:
        est = {int(f): p for f, p in zip(frames, P)}
    else:
        kc = {k: cfg[k] for k in ('mode', 'q', 'lam', 'smin', 'rfix', 'smooth') if k in cfg}
        est = refine_track(frames, P, sigmas, max_gap=max_gap, **kc)
    if bones is None or bones.get('order', 'after') != 'after' or not est:
        return est
    fs = np.array(sorted(est))
    X = np.stack([est[f] for f in fs])
    meas = {int(f): k for k, f in enumerate(frames)}
    med = np.median(sigmas, axis=0)
    S = np.stack([sigmas[meas[f]] if f in meas else med for f in fs])   # gap frames: typical sigma
    Y = _bone_stage(X, S, poses, sigmas, frames, fs, bones, smooth, parents, order, sym_pairs)
    return {int(f): y for f, y in zip(fs, Y)}


# ---------------------------------------------------------------------------
# Standard smoothers used for comparison (applied to runs of consecutive frames; they do not fill gaps)
# ---------------------------------------------------------------------------
def runs(frames):
    frames = np.asarray(frames)
    if len(frames) == 0:
        return []
    cut = np.nonzero(np.diff(frames) != 1)[0] + 1
    return np.split(np.arange(len(frames)), cut)


def one_euro(Z, freq, min_cutoff, beta, d_cutoff=1.0):
    # One-Euro filter (Casiez et al., CHI 2012), vectorized over columns. Z: (T, D).
    def alpha(cutoff):
        tau = 1.0 / (2 * np.pi * cutoff)
        return 1.0 / (1.0 + tau * freq)
    Y = np.empty_like(Z)
    Y[0] = Z[0]
    dx_prev = np.zeros(Z.shape[1])
    ad = alpha(d_cutoff)
    for t in range(1, len(Z)):
        dx = (Z[t] - Y[t - 1]) * freq
        dx_prev = ad * dx + (1 - ad) * dx_prev
        a = alpha(min_cutoff + beta * np.abs(dx_prev))
        Y[t] = a * Z[t] + (1 - a) * Y[t - 1]
    return Y


def smoothnet_forward(W, x):
    # SmoothNet (Zeng et al., ECCV 2022) in eval mode; x: (N, C, T) -> (N, C, T).
    lrelu = lambda v, s: np.where(v > 0, v, s * v)
    h = lrelu(x @ W['enc_w'].T + W['enc_b'], 0.1)
    for (w1, b1, w2, b2) in W['blocks']:
        r = lrelu(h @ w1.T + b1, 0.2)
        r = lrelu(r @ w2.T + b2, 0.2)
        h = h + r
    return h @ W['dec_w'].T + W['dec_b']


def smoothnet_weights(state):
    # Convert a SmoothNet state dict (numpy arrays) into the structure used above.
    state = {k.replace('module.', ''): np.asarray(v, np.float32) for k, v in state.items()}
    nb = len({k.split('.')[1] for k in state if k.startswith('res_blocks.')})
    blocks = [(state[f'res_blocks.{i}.linear1.weight'], state[f'res_blocks.{i}.linear1.bias'],
               state[f'res_blocks.{i}.linear2.weight'], state[f'res_blocks.{i}.linear2.bias']) for i in range(nb)]
    return {'enc_w': state['encoder.0.weight'], 'enc_b': state['encoder.0.bias'], 'blocks': blocks,
            'dec_w': state['decoder.weight'], 'dec_b': state['decoder.bias'],
            'window': int(state['encoder.0.weight'].shape[1])}


def smoothnet_apply(W, Z, chunk=256):
    # Z: (T, C) in metres. Sliding windows with step 1, overlapping outputs averaged (as in SmoothNet).
    T, C = Z.shape
    win = W['window']
    Zp = np.concatenate([Z, np.repeat(Z[-1:], win - T, 0)]) if T < win else Z
    nW = len(Zp) - win + 1
    acc = np.zeros(Zp.shape); cnt = np.zeros((len(Zp), 1))
    for s0 in range(0, nW, chunk):
        idx = np.arange(s0, min(nW, s0 + chunk))
        win_in = np.stack([Zp[i:i + win].T for i in idx]).astype(np.float32)    # (n, C, win)
        out = smoothnet_forward(W, win_in)
        for j in range(win):
            acc[idx + j] += out[:, :, j]
            cnt[idx + j] += 1
    return (acc / cnt)[:T]


def apply_smoother(frames, poses, method, p, fps=30.0, snet=None):
    # Returns dict frame -> pose for the measured frames only.
    out = {}
    X = poses.reshape(len(poses), -1)
    frames = np.asarray(frames)
    for idx in runs(frames):
        Z = X[idx]
        if method == 'oneeuro':
            Y = one_euro(Z, fps, p['min_cutoff'], p['beta'])
        elif method == 'savgol':
            w = min(p['window'], len(Z) if len(Z) % 2 else len(Z) - 1)
            Y = savgol_filter(Z, w, p['order'], axis=0, mode='interp') if w > p['order'] else Z
        elif method == 'gauss':
            Y = gaussian_filter1d(Z, p['sigma'], axis=0, mode='nearest')
        elif method == 'smoothnet':
            Y = smoothnet_apply(snet, Z / 1000.0) * 1000.0
        else:
            raise ValueError(method)
        for k, f in enumerate(frames[idx]):
            out[int(f)] = Y[k].reshape(-1, 3)
    return out


# ---------------------------------------------------------------------------
# Simple tracking-by-detection (box overlap + 2D pose distance, Hungarian assignment)
# ---------------------------------------------------------------------------
def box_iou(a, b):
    # a: (4,), b: (m, 4) as [x, y, w, h]
    x0 = np.maximum(a[0], b[:, 0]); y0 = np.maximum(a[1], b[:, 1])
    x1 = np.minimum(a[0] + a[2], b[:, 0] + b[:, 2]); y1 = np.minimum(a[1] + a[3], b[:, 1] + b[:, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    return inter / (a[2] * a[3] + b[:, 2] * b[:, 3] - inter + 1e-9)


def track_detections(frame_list, boxes, kps, iou_thr=0.3, kp_thr=0.5, max_age=15):
    # frame_list: sorted frame indices; boxes[f]: (n, 4); kps[f]: (n, J, 2).
    # Returns ids[f]: array with a track id per detection.
    active, ids, next_id = {}, {}, 0
    for f in frame_list:
        B, KP = boxes[f], kps[f]
        idf = -np.ones(len(B), int)
        cand = [t for t, s in active.items() if f - s['last'] <= max_age + 1]
        if cand and len(B):
            C = np.zeros((len(cand), len(B))); M = np.zeros_like(C, bool)
            for i, t in enumerate(cand):
                s = active[t]
                iou = box_iou(s['box'], B)
                d = np.linalg.norm(s['kp'][None] - KP, axis=-1).mean(-1) / max(s['box'][3], 1.0)
                C[i] = (1 - iou) + d
                M[i] = (iou > iou_thr) | (d < kp_thr)
            for r, c in zip(*linear_sum_assignment(C)):
                if M[r, c]:
                    idf[c] = cand[r]
        for c in range(len(B)):
            if idf[c] < 0:
                idf[c] = next_id; next_id += 1
            active[idf[c]] = {'last': f, 'box': B[c], 'kp': KP[c]}
        for t in [t for t, s in active.items() if f - s['last'] > max_age]:
            del active[t]
        ids[f] = idf
    return ids


# ---------------------------------------------------------------------------
# Real-object occluders (Pascal VOC), following Sarandi et al. (2018)
# ---------------------------------------------------------------------------
def load_voc_occluders(voc_root, max_count=None):
    import os, xml.etree.ElementTree as ET
    import PIL.Image, cv2
    occ = []
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (8, 8))
    ann_dir = os.path.join(voc_root, 'Annotations')
    for name in sorted(os.listdir(ann_dir)):
        root = ET.parse(os.path.join(ann_dir, name)).getroot()
        if root.find('segmented').text == '0':
            continue
        boxes = []
        for i, obj in enumerate(root.findall('object')):
            if obj.find('name').text == 'person' or obj.find('difficult').text != '0' or obj.find('truncated').text != '0':
                continue
            bb = obj.find('bndbox')
            boxes.append((i, [int(float(bb.find(s).text)) for s in ('xmin', 'ymin', 'xmax', 'ymax')], obj.find('name').text))
        if not boxes:
            continue
        fn = root.find('filename').text
        im = np.asarray(PIL.Image.open(os.path.join(voc_root, 'JPEGImages', fn)).convert('RGB'))
        lab = np.asarray(PIL.Image.open(os.path.join(voc_root, 'SegmentationObject', fn.replace('.jpg', '.png'))))
        for i, (x0, y0, x1, y1), cls in boxes:
            m = (lab[y0:y1, x0:x1] == i + 1).astype(np.uint8) * 255
            if cv2.countNonZero(m) < 500:
                continue
            er = cv2.erode(m, se)
            m[er < m] = 192
            rgba = np.concatenate([im[y0:y1, x0:x1], m[..., None]], -1)
            rgba = cv2.resize(rgba, (max(1, rgba.shape[1] // 2), max(1, rgba.shape[0] // 2)), interpolation=cv2.INTER_AREA)
            occ.append({'rgba': rgba, 'cls': cls, 'area': float((rgba[..., 3] > 0).sum())})
            if max_count and len(occ) >= max_count:
                return occ
    return occ


def paste_occluder(img, occ, box, frac, center_uv, mask_acc=None):
    # Scale the object so that its visible area is `frac` of the box area and paste it with its centre at
    # box-relative position center_uv (in [0,1]^2). Returns the new image and the object mask (bool, H x W).
    import cv2
    x, y, w, h = box
    s = np.sqrt(max(frac, 1e-6) * w * h / max(occ['area'], 1.0))
    rgba = occ['rgba']
    nw, nh = max(1, int(round(rgba.shape[1] * s))), max(1, int(round(rgba.shape[0] * s)))
    rgba = cv2.resize(rgba, (nw, nh), interpolation=cv2.INTER_LINEAR if s > 1 else cv2.INTER_AREA)
    H, W = img.shape[:2]
    cx, cy = x + center_uv[0] * w, y + center_uv[1] * h
    x0, y0 = int(round(cx - nw / 2)), int(round(cy - nh / 2))
    a0, b0 = max(0, x0), max(0, y0)
    a1, b1 = min(W, x0 + nw), min(H, y0 + nh)
    out = img.copy()
    mask = np.zeros((H, W), bool) if mask_acc is None else mask_acc
    if a1 > a0 and b1 > b0:
        src = rgba[b0 - y0:b1 - y0, a0 - x0:a1 - x0]
        al = src[..., 3:].astype(np.float32) / 255.0
        out[b0:b1, a0:a1] = (al * src[..., :3] + (1 - al) * out[b0:b1, a0:a1]).astype(np.uint8)
        mask[b0:b1, a0:a1] |= src[..., 3] >= 128
    return out, mask


def joints_on_mask(uv, mask):
    H, W = mask.shape
    u = np.clip(np.round(uv[:, 0]).astype(int), 0, W - 1)
    v = np.clip(np.round(uv[:, 1]).astype(int), 0, H - 1)
    ok = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    return ok & mask[v, u]


# ---------------------------------------------------------------------------
# Statistics
# ---------------------------------------------------------------------------
def paired_tests(a, b, n_boot=10000, weights=None, seed=0):
    # a, b: per-sequence means of two methods (same sequences). Wilcoxon signed-rank test and a
    # bootstrap 95% interval (resampling sequences) of the pose-weighted mean difference b - a.
    from scipy.stats import wilcoxon
    a, b = np.asarray(a, float), np.asarray(b, float)
    w = np.ones_like(a) if weights is None else np.asarray(weights, float)
    d = b - a
    try:
        p = float(wilcoxon(a, b).pvalue)
    except ValueError:
        p = float('nan')
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot = (d[idx] * w[idx]).sum(1) / w[idx].sum(1)
    return {'mean_diff': float((d * w).sum() / w.sum()), 'ci95': [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
            'wilcoxon_p': p, 'n_sequences': int(len(d)), 'n_improved': int((d < 0).sum())}


def spearman(x, y):
    def rank(v):
        r = np.empty(len(v)); r[np.argsort(v)] = np.arange(len(v)); return r
    return float(np.corrcoef(rank(x), rank(y))[0, 1])


import time
import numpy as np

# ---------------------------------------------------------------------------
# Preparing inference records
# rec = {'seq', 'split', 'model', 'fps', 'frames': [{'frame', 'boxes' (n,5), 'p3' (n,A,J,3), 'p2' (n,J,2), 't',
#        'gts': [{'actor', 'gt3d' (J,3), 'gt2d' (J,2), 'inside' (J,), optional 'occl' (J,)}]}]}
# ---------------------------------------------------------------------------
def prep_record(rec, root, aug_idx=None):
    frames = []
    for fr in rec['frames']:
        p3 = np.asarray(fr['p3'], np.float64)
        if p3.ndim == 3:
            p3 = p3[:, None]
        if aug_idx is not None:
            p3 = p3[:, aug_idx]
        dets = []
        for k in range(len(p3)):
            m, s = rootrel_mean_and_sigma(p3[k], root=root)
            dets.append({'pose': m, 'sigma': s, 'kp': np.asarray(fr['p2'][k], np.float64),
                         'box': np.asarray(fr['boxes'][k][:4], np.float64)})
        frames.append({'frame': int(fr['frame']), 'dets': dets, 'gts': fr['gts']})
    return {'seq': rec['seq'], 'fps': float(rec.get('fps', 30.0)), 'frames': frames}


def gt_identity_tracks(preps, root):
    # One track per annotated person: in every frame the detection closest to that person (2D) is used.
    tracks = []
    for P in preps:
        by_actor = {}
        for fr in P['frames']:
            if not fr['gts']:
                continue
            kp = np.array([d['kp'] for d in fr['dets']]) if fr['dets'] else np.zeros((0, 1, 2))
            assign = match_detections(kp, [np.asarray(g['gt2d'], np.float64) for g in fr['gts']])
            for g, d in zip(fr['gts'], assign):
                tr = by_actor.setdefault(g['actor'], {'seq': P['seq'], 'fps': P['fps'], 'actor': g['actor'], 'gt': {},
                                                      'inside': {}, 'occl': {}, 'frames': [], 'pose': [], 'sigma': []})
                g3 = np.asarray(g['gt3d'], np.float64)
                tr['gt'][fr['frame']] = g3 - g3[root]
                tr['inside'][fr['frame']] = np.asarray(g['inside'], bool)
                if g.get('occl') is not None:
                    tr['occl'][fr['frame']] = np.asarray(g['occl'], bool)
                if d is not None:
                    tr['frames'].append(fr['frame']); tr['pose'].append(fr['dets'][d]['pose']); tr['sigma'].append(fr['dets'][d]['sigma'])
        for tr in by_actor.values():
            J = len(next(iter(tr['gt'].values())))
            tr['frames'] = np.array(tr['frames'], int)
            tr['pose'] = np.array(tr['pose']).reshape(-1, J, 3)
            tr['sigma'] = np.array(tr['sigma']).reshape(-1, J)
            tracks.append(tr)
    return tracks


def run_method(frames, poses, sigmas, fps, spec, ctx):
    kind = spec['kind']
    if kind == 'raw':
        return {int(f): p for f, p in zip(frames, poses)}
    if kind == 'refine':
        return refine_v4(frames, poses, sigmas, spec['cfg'], ctx['parents'], ctx['order'], ctx['sym'], ctx['max_gap'])
    if kind == 'smoother':
        return apply_smoother(frames, poses, spec['method'], spec['params'], fps=fps, snet=ctx.get('snet'))
    raise ValueError(kind)


def _align_err(est, gt, ctx):
    # Root-aligned errors on the full skeleton; Procrustes alignment on the evaluated joints.
    e, epa = joint_errors_r(est, gt, ctx['root'])
    J = ctx['eval_joints']
    if J is None:
        return e, epa
    return e[J], np.linalg.norm(similarity_align(est[J], gt[J]) - gt[J], axis=1)


def evaluate_tracks(tracks, spec, ctx, fast=False):
    # Errors on frames where the person was detected; gap-filled frames are reported separately.
    per_seq = {}
    errs, errs_pa, accs, gap_errs, occ_e, vis_e, out_e = [], [], [], [], [], [], []
    n_valid = n_matched = 0
    J = ctx['eval_joints']
    for tr in tracks:
        est = run_method(tr['frames'], tr['pose'], tr['sigma'], tr['fps'], spec, ctx)
        matched = [int(f) for f in tr['frames']]
        mset = set(matched)
        n_valid += len(tr['gt']); n_matched += len(matched)
        S = per_seq.setdefault(tr['seq'], {'sum': 0.0, 'n': 0, 'acc_sum': 0.0, 'acc_n': 0})
        for f in matched:
            if fast:
                e = np.linalg.norm(est[f] - tr['gt'][f], axis=1)
                errs.append(e if J is None else e[J])
                continue
            e, epa = _align_err(est[f], tr['gt'][f], ctx)
            errs.append(e); errs_pa.append(epa)
            S['sum'] += e.mean(); S['n'] += 1
            ins = tr['inside'][f] if J is None else tr['inside'][f][J]
            out_e.append(e[~ins]); vis_e.append(e[ins])
            if f in tr['occl']:
                oc = tr['occl'][f] if J is None else tr['occl'][f][J]
                occ_e.append((e[oc], e[~oc]))
        if fast:
            continue
        for f, g in tr['gt'].items():
            if f not in mset and f in est:
                gap_errs.append(np.linalg.norm(est[f] - g, axis=1).mean())
        run = []
        for f in matched + [None]:
            if f is not None and (not run or f == run[-1] + 1):
                run.append(f); continue
            if len(run) >= 3:
                a = accel_error(np.stack([est[x] for x in run]), np.stack([tr['gt'][x] for x in run]))
                accs.append(a); S['acc_sum'] += a.sum(); S['acc_n'] += len(a)
            run = [f] if f is not None else []
    if fast:
        return float(np.concatenate(errs).mean())
    res = summarize_r(np.stack(errs), np.stack(errs_pa), ctx['pck_thr'], ctx['auc_max'], ctx.get('pck_joints'))
    res['Accel'] = float(np.concatenate(accs).mean()) if accs else None
    res['valid'] = int(n_valid); res['evaluated'] = int(n_matched); res['filled'] = int(len(gap_errs))
    res['filled_MPJPE'] = float(np.mean(gap_errs)) if gap_errs else None
    res['coverage_pct'] = 100.0 * (n_matched + len(gap_errs)) / max(n_valid, 1)
    oe = np.concatenate(out_e) if out_e else np.zeros(0)
    res['MPJPE_outside_image'] = float(oe.mean()) if len(oe) else None
    if occ_e:
        a = np.concatenate([x for x, _ in occ_e]); b = np.concatenate([y for _, y in occ_e])
        res['MPJPE_occluded_joints'] = float(a.mean()) if len(a) else None
        res['MPJPE_visible_joints'] = float(b.mean()) if len(b) else None
        res['occluded_joint_pct'] = float(100 * len(a) / max(len(a) + len(b), 1))
    res['per_seq'] = {s: {'MPJPE': v['sum'] / max(v['n'], 1), 'n': v['n'],
                          'Accel': v['acc_sum'] / v['acc_n'] if v['acc_n'] else None, 'acc_n': v['acc_n']} for s, v in per_seq.items()}
    return res


def evaluate_with_tracker(preps, spec, ctx, iou_thr=0.3, kp_thr=0.5):
    # Realistic setting: people are linked over time by a tracker, not by the ground-truth identities.
    errs, errs_pa, accs, gap_e = [], [], [], []
    n_valid = n_det = n_fill = switches = 0
    n_tracks = 0
    for P in preps:
        fl = [fr['frame'] for fr in P['frames']]
        boxes = {fr['frame']: np.array([d['box'] for d in fr['dets']]).reshape(-1, 4) for fr in P['frames']}
        kps = {fr['frame']: (np.array([d['kp'] for d in fr['dets']]) if fr['dets'] else np.zeros((0, 1, 2))) for fr in P['frames']}
        ids = track_detections(fl, boxes, kps, iou_thr, kp_thr, ctx['max_gap'])
        tr = {}
        for fr in P['frames']:
            for d, t in zip(fr['dets'], ids[fr['frame']]):
                x = tr.setdefault(int(t), {'frames': [], 'pose': [], 'sigma': [], 'kp': []})
                x['frames'].append(fr['frame']); x['pose'].append(d['pose']); x['sigma'].append(d['sigma']); x['kp'].append(d['kp'])
        n_tracks += len(tr)
        cand = {}
        for t, x in tr.items():
            F = np.array(x['frames']); KP = np.array(x['kp'])
            est = run_method(F, np.array(x['pose']), np.array(x['sigma']), P['fps'], spec, ctx)
            fset = set(F.tolist())
            for f, pose in est.items():
                if f in fset:
                    kp = KP[np.searchsorted(F, f)]
                else:  # 2D position of a filled pose: linear interpolation (only used for matching)
                    kp = np.stack([np.stack([np.interp(f, F, KP[:, j, c]) for c in range(2)]) for j in range(KP.shape[1])])
                cand.setdefault(f, []).append((t, pose, kp, f in fset))
        last = {}
        actor_seq = {}
        for fr in P['frames']:
            f = fr['frame']
            if not fr['gts']:
                continue
            gts2d = [np.asarray(g['gt2d'], np.float64) for g in fr['gts']]
            n_valid += len(gts2d)
            C = cand.get(f, [])
            det = [c for c in C if c[3]]; fil = [c for c in C if not c[3]]
            a1 = match_detections(np.array([c[2] for c in det]).reshape(-1, gts2d[0].shape[0], 2), gts2d)
            rem = [i for i, a in enumerate(a1) if a is None]
            a2 = match_detections(np.array([c[2] for c in fil]).reshape(-1, gts2d[0].shape[0], 2), [gts2d[i] for i in rem]) if rem and fil else [None] * len(rem)
            final = list(a1)
            for i, a in zip(rem, a2):
                final[i] = ('f', a) if a is not None else None
            for g, a in zip(fr['gts'], final):
                if a is None:
                    continue
                c = fil[a[1]] if isinstance(a, tuple) else det[a]
                g3 = np.asarray(g['gt3d'], np.float64); gt = g3 - g3[ctx['root']]
                if c[3]:
                    e, epa = _align_err(c[1], gt, ctx); errs.append(e); errs_pa.append(epa); n_det += 1
                else:
                    gap_e.append(np.linalg.norm(c[1] - gt, axis=1).mean()); n_fill += 1
                key = (P['seq'], g['actor'])
                if key in last and last[key] != c[0]:
                    switches += 1
                last[key] = c[0]
                if c[3]:
                    actor_seq.setdefault(key, []).append((f, c[1], gt))
        for key, L in actor_seq.items():
            run = []
            for item in L + [None]:
                if item is not None and (not run or item[0] == run[-1][0] + 1):
                    run.append(item); continue
                if len(run) >= 3:
                    accs.append(accel_error(np.stack([r[1] for r in run]), np.stack([r[2] for r in run])))
                run = [item] if item is not None else []
    res = summarize_r(np.stack(errs), np.stack(errs_pa), ctx['pck_thr'], ctx['auc_max'], ctx.get('pck_joints'))
    res['Accel'] = float(np.concatenate(accs).mean()) if accs else None
    res['valid'] = int(n_valid); res['evaluated'] = int(n_det); res['filled'] = int(n_fill)
    res['filled_MPJPE'] = float(np.mean(gap_e)) if gap_e else None
    res['coverage_pct'] = 100.0 * (n_det + n_fill) / max(n_valid, 1)
    res['id_switches'] = int(switches); res['tracks'] = int(n_tracks)
    return res


def grid_search(tracks, specs, ctx, desc=''):
    out = []
    for sp in specs:
        out.append((evaluate_tracks(tracks, sp, ctx, fast=True), sp))
    out.sort(key=lambda x: x[0])
    return out


def uncertainty_analysis(tracks, root):
    S, E, IN = [], [], []
    keep = None
    for tr in tracks:
        J = tr['pose'].shape[1]
        keep = [j for j in range(J) if j != root]
        for f, P, sg in zip(tr['frames'], tr['pose'], tr['sigma']):
            S.append(sg[keep]); E.append(np.linalg.norm(P - tr['gt'][f], axis=1)[keep]); IN.append(tr['inside'][f][keep])
    S, E, IN = np.concatenate(S), np.concatenate(E), np.concatenate(IN)
    q = np.quantile(S, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
    bins = np.clip(np.searchsorted(q, S, side='right') - 1, 0, 4)
    return {'spearman_sigma_error': spearman(S, E), 'n_joints': int(len(S)),
            'joints_inside_image': {'count': int(IN.sum()), 'mean_error': float(E[IN].mean()), 'mean_sigma': float(S[IN].mean())},
            'joints_outside_image': {'count': int((~IN).sum()), 'mean_error': float(E[~IN].mean()) if (~IN).any() else None,
                                     'mean_sigma': float(S[~IN].mean()) if (~IN).any() else None},
            'error_by_sigma_quintile': [float(E[bins == b].mean()) for b in range(5)],
            'sigma_quintile_edges': [float(v) for v in q]}


# ---------------------------------------------------------------------------
# Occlusion episodes (temporal study) and the per-frame occluder study
# ---------------------------------------------------------------------------
def episode_tracks(records, root):
    # records: list over sequences of {'seq', 'fps', 'items': [{'frame', 'actor', 'pred_aug', 'gt3d', 'episode', 'affected'}]}
    tracks = []
    for R in records:
        by = {}
        for it in R['items']:
            by.setdefault(it['actor'], []).append(it)
        for a, its in by.items():
            its.sort(key=lambda x: x['frame'])
            fr, po, sg, gt, ep, af, ins = [], [], [], {}, {}, {}, {}
            for it in its:
                m, s = rootrel_mean_and_sigma(np.asarray(it['pred_aug'], np.float64), root=root)
                g3 = np.asarray(it['gt3d'], np.float64)
                fr.append(it['frame']); po.append(m); sg.append(s)
                gt[it['frame']] = g3 - g3[root]; ep[it['frame']] = bool(it['episode']); af[it['frame']] = np.asarray(it['affected'], bool)
                ins[it['frame']] = np.ones(len(g3), bool)
            tracks.append({'seq': R['seq'], 'fps': R.get('fps', 30.0), 'actor': a, 'frames': np.array(fr), 'pose': np.array(po),
                           'sigma': np.array(sg), 'gt': gt, 'episode': ep, 'affected': af, 'inside': ins, 'occl': {}})
    return tracks


def evaluate_episodes(tracks, spec, ctx):
    e_ep, e_clean, e_aff, e_unaff, s_aff, s_unaff = [], [], [], [], [], []
    per_seq = {}
    for tr in tracks:
        est = run_method(tr['frames'], tr['pose'], tr['sigma'], tr['fps'], spec, ctx)
        sig = {int(f): s for f, s in zip(tr['frames'], tr['sigma'])}
        S = per_seq.setdefault(tr['seq'], {'sum': 0.0, 'n': 0})
        for f in tr['frames']:
            f = int(f)
            e = np.linalg.norm(est[f] - tr['gt'][f], axis=1)
            if tr['episode'][f]:
                e_ep.append(e.mean()); S['sum'] += e.mean(); S['n'] += 1
                a = tr['affected'][f].copy(); a[ctx['root']] = False
                u = ~a; u[ctx['root']] = False
                e_aff.append(e[a]); e_unaff.append(e[u]); s_aff.append(sig[f][a]); s_unaff.append(sig[f][u])
            else:
                e_clean.append(e.mean())
    cat = lambda L: np.concatenate(L) if L else np.zeros(0)
    A, U = cat(e_aff), cat(e_unaff)
    return {'MPJPE_occluded_frames': float(np.mean(e_ep)) if e_ep else None, 'MPJPE_clean_frames': float(np.mean(e_clean)) if e_clean else None,
            'MPJPE_occluded_joints': float(A.mean()) if len(A) else None, 'MPJPE_other_joints_in_occluded_frames': float(U.mean()) if len(U) else None,
            'sigma_occluded_joints': float(cat(s_aff).mean()) if len(A) else None, 'sigma_other_joints': float(cat(s_unaff).mean()) if len(U) else None,
            'occluded_frames': len(e_ep), 'clean_frames': len(e_clean),
            'per_seq': {s: {'MPJPE': v['sum'] / max(v['n'], 1), 'n': v['n']} for s, v in per_seq.items()}}


def robust_summary(records, root):
    # records: [{'kind', 'level', 'pred', 'sigma', 'gt', 'affected'}] -> table rows (as in the v3 study)
    out = {}
    keys = sorted({(r['kind'], r['level']) for r in records})
    for kind, lev in keys:
        rs = [r for r in records if r['kind'] == kind and r['level'] == lev]
        E = [joint_errors_r(np.asarray(r['pred'], np.float64), np.asarray(r['gt'], np.float64), root) for r in rs]
        e_all = np.stack([e for e, _ in E]); e_pa = np.stack([p for _, p in E])
        A = np.stack([r['affected'] for r in rs]).astype(bool); A[:, root] = False
        V = ~A; V[:, root] = False
        Sg = np.stack([r['sigma'] for r in rs])
        keep = [j for j in range(A.shape[1]) if j != root]
        out[f'{kind}_{int(round(lev * 100))}'] = {
            'poses': len(rs), 'MPJPE': float(e_all.mean()), 'PA-MPJPE': float(e_pa.mean()),
            'affected_joint_pct': float(100 * A[:, keep].mean()),
            'MPJPE_affected_joints': float(e_all[A].mean()) if A.any() else None,
            'MPJPE_other_joints': float(e_all[V].mean()),
            'sigma_affected_joints': float(Sg[A].mean()) if A.any() else None,
            'sigma_other_joints': float(Sg[V].mean())}
    return out


def time_method(tracks, spec, ctx):
    n = sum(len(tr['frames']) for tr in tracks)
    t0 = time.perf_counter()
    for tr in tracks:
        run_method(tr['frames'], tr['pose'], tr['sigma'], tr['fps'], spec, ctx)
    return 1000.0 * (time.perf_counter() - t0) / max(n, 1)


# ---- self-tests ----
_rng = np.random.default_rng(0)
_gt = _rng.normal(size=(24, 3)) * 300
_c, _s = np.cos(0.7), np.sin(0.7)
_R = np.array([[_c, -_s, 0], [_s, _c, 0], [0, 0, 1]])
assert joint_errors(1.2 * _gt @ _R.T + 50, _gt)[1].mean() < 1e-6
_o = topo_order(SMPL_PARENTS); assert all(_o.index(SMPL_PARENTS[j]) < _o.index(j) for j in range(1, 24))
_o = topo_order(MPI17_PARENTS); assert all(_o.index(MPI17_PARENTS[j]) < _o.index(j) for j in range(17) if j != 14)
_X = _rng.normal(size=(50, 24, 3)) * 200; _X -= _X[:, :1]
_L = estimate_bones(_X, np.full((50, 24), 10.0), SMPL_PARENTS)
_Y = apply_bones(_X, _L, SMPL_PARENTS, topo_order(SMPL_PARENTS), np.full((50, 24), 10.0), 1.0, 3.0, iters=30)
assert np.nanmax(np.abs(bone_lengths(_Y, SMPL_PARENTS)[:, 1:] - _L[1:])) < 5.0, 'bone projection self-test failed'
print('Helper self-tests passed.')

## Step 4 — Download 3DPW into Colab (needed for Parts A, B and D)

In [ ]:
NEED_3DPW = RUN_3DPW_MAIN or RUN_VOC_STUDY or RUN_SECOND_MODEL or RUN_HMR2
DL_DIR, LOCAL = '/content/3DPW_download', '/content/3DPW'
if NEED_3DPW:
    os.makedirs(DL_DIR, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
    for _name in ('sequenceFiles.zip', 'imageFiles.zip'):
        _dst = os.path.join(DL_DIR, _name)
        if not os.path.exists(_dst):
            print('Downloading', _name)
            subprocess.run(['wget', '-q', '--show-progress', '--progress=bar:force', '-c', '-O', _dst + '.part', DATA_URL + _name], check=True)
            os.rename(_dst + '.part', _dst)
    if not glob.glob(f'{LOCAL}/**/test/*.pkl', recursive=True):
        with zipfile.ZipFile(os.path.join(DL_DIR, 'sequenceFiles.zip')) as z:
            z.extractall(LOCAL)
    def _split_files(split):
        return {os.path.basename(p)[:-4]: p for p in sorted(glob.glob(f'{LOCAL}/**/{split}/*.pkl', recursive=True))
                if '__MACOSX' not in p and not os.path.basename(p).startswith('._')}
    _test, _val = _split_files('test'), _split_files('validation')
    SEQ_FILES = {**_test, **_val}
    TEST_SEQS, VAL_SEQS = sorted(_test), sorted(_val)
    if QUICK_TEST:
        TEST_SEQS, VAL_SEQS = TEST_SEQS[:1], VAL_SEQS[:1]
    SEQ_INDEX = {name: k for k, name in enumerate(sorted(SEQ_FILES))}
    IMG_DIRS = {}
    with zipfile.ZipFile(os.path.join(DL_DIR, 'imageFiles.zip')) as z:
        all_names = z.namelist()
        for s in TEST_SEQS + VAL_SEQS:
            members = sorted(n for n in all_names if n.endswith('.jpg') and f'/{s}/' in '/' + n
                             and '__MACOSX' not in n and not os.path.basename(n).startswith('._'))
            if not (os.path.exists(os.path.join(LOCAL, members[0])) and os.path.exists(os.path.join(LOCAL, members[-1]))):
                for n in tqdm(members, desc=f'unzip {s}', leave=False):
                    z.extract(n, LOCAL)
            IMG_DIRS[s] = os.path.dirname(os.path.join(LOCAL, members[0]))
    print(len(TEST_SEQS), 'test and', len(VAL_SEQS), 'validation sequences ready')
else:
    print('3DPW not needed for the selected parts.')

## Step 5 — Model download, loading and checks

In [ ]:
MODELS = {}
def load_metrabs(path):
    # The 2021 SavedModels store a duplicated argument name ('training') that newer TensorFlow rejects.
    try:
        return tf.saved_model.load(path)
    except ValueError as e:
        if 'duplicate parameter name' not in str(e):
            raise
        import inspect
        _orig_init = inspect.Signature.__init__
        def _dedup_init(self, parameters=None, *args, **kwargs):
            if parameters is not None:
                seen, kept = set(), []
                for p in parameters:
                    if p.name not in seen:
                        seen.add(p.name); kept.append(p)
                parameters = kept
            _orig_init(self, parameters, *args, **kwargs)
        inspect.Signature.__init__ = _dedup_init
        return tf.saved_model.load(path)

def get_model(name):
    if name in MODELS:
        return MODELS[name]
    for other in list(MODELS):      # keep only one model on the GPU
        del MODELS[other]
    gc.collect()
    os.makedirs('/content/models', exist_ok=True)
    zpath = f'/content/models/{name}.zip'
    if not os.path.exists(zpath):
        print('Downloading model', name)
        urllib.request.urlretrieve(f'https://omnomnom.vision.rwth-aachen.de/data/metrabs/{name}.zip', zpath + '.part')
        os.rename(zpath + '.part', zpath)
    root = f'/content/models/{name}'
    if not glob.glob(f'{root}/**/saved_model.pb', recursive=True):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(root)
    m = load_metrabs(os.path.dirname(glob.glob(f'{root}/**/saved_model.pb', recursive=True)[0]))
    names = [n.decode() for n in m.per_skeleton_joint_names['smpl_24'].numpy()]
    if [n.split('_')[0] for n in names] != SMPL_JOINTS:
        raise RuntimeError('Unexpected smpl_24 joint order in ' + name + ': ' + str(names))
    MODELS[name] = m
    print('Loaded', name)
    return m

if NEED_3DPW:
    _first = load_sequence(SEQ_FILES[TEST_SEQS[0]])
    sc = camera_convention_scores(_first)
    INVERT_CAM_POSES = sc[True] < sc[False]
    print(f'Camera check (pixels): as given = {sc[False]:.1f}, inverted = {sc[True]:.1f} -> using', 'INVERTED' if INVERT_CAM_POSES else 'AS GIVEN')

## Part A — Run the ResNet-50 model on every 3DPW test and validation frame
All detections of every frame are saved (for the tracker), with all 5 crop predictions (for the uncertainty).

In [ ]:
def run_3dpw(model_name, split, seq_name):
    out = os.path.join(RESULTS_DIR, f'inf3dpw_{model_name}_{split}_{seq_name}.pkl')
    if os.path.exists(out):
        return
    model = get_model(model_name)
    seq = load_sequence(SEQ_FILES[seq_name])
    K = np.asarray(seq['cam_intrinsics'], dtype=np.float64); K_tf = tf.constant(K, tf.float32)
    n = len(seq['cam_poses'])
    recs = []
    for i in tqdm(range(min(n, 150) if QUICK_TEST else n), desc=f'{model_name} | {split} | {seq_name}'):
        image = tf.image.decode_jpeg(tf.io.read_file(os.path.join(IMG_DIRS[seq_name], f'image_{i:05d}.jpg')))
        H, W = int(image.shape[0]), int(image.shape[1])
        gts = []
        for a in range(len(seq['jointPositions'])):
            if not frame_is_valid(seq, a, i):
                continue
            Xc = to_camera(np.asarray(seq['jointPositions'][a]).reshape(-1, 24, 3)[i], np.asarray(seq['cam_poses'][i]), INVERT_CAM_POSES) * 1000.0
            uv = project(Xc, K)
            inside = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H) & (Xc[:, 2] > 0)
            gts.append({'actor': a, 'gt3d': Xc.astype(np.float32), 'gt2d': uv.astype(np.float32), 'inside': inside})
        t0 = time.time()
        pred = model.detect_poses(image, intrinsic_matrix=K_tf, skeleton='smpl_24', num_aug=NUM_AUG, average_aug=False)
        p3, p2, bx = pred['poses3d'].numpy(), pred['poses2d'].numpy(), pred['boxes'].numpy()
        dt = time.time() - t0
        if p3.ndim == 3:
            p3, p2 = p3[:, None], p2[:, None]
        recs.append({'frame': i, 'boxes': bx.astype(np.float32), 'p3': p3.astype(np.float32),
                     'p2': p2.mean(1).astype(np.float32), 't': dt, 'gts': gts})
    save_pkl({'seq': seq_name, 'split': split, 'model': model_name, 'fps': 30.0, 'frames': recs}, out)

def measure_speed(model_name, n=40):
    out = os.path.join(RESULTS_DIR, f'speed_{model_name}.json')
    if os.path.exists(out):
        return
    model = get_model(model_name)
    s = TEST_SEQS[0]; seq = load_sequence(SEQ_FILES[s])
    K_tf = tf.constant(np.asarray(seq['cam_intrinsics'], np.float32))
    imgs = [tf.image.decode_jpeg(tf.io.read_file(os.path.join(IMG_DIRS[s], f'image_{i:05d}.jpg'))) for i in range(n + 5)]
    res = {'gpu': str(tf.config.list_physical_devices('GPU')), 'frames': n}
    for A in (1, 3, 5):
        for im in imgs[:5]:
            model.detect_poses(im, intrinsic_matrix=K_tf, skeleton='smpl_24', num_aug=A)
        t0 = time.time(); persons = 0
        for im in imgs[5:]:
            persons += int(model.detect_poses(im, intrinsic_matrix=K_tf, skeleton='smpl_24', num_aug=A)['poses3d'].shape[0])
        res[f'ms_per_frame_aug{A}'] = 1000 * (time.time() - t0) / n
        res['persons_per_frame'] = persons / n
    save_json(res, out)
    print(model_name, res)

if RUN_3DPW_MAIN:
    for s in TEST_SEQS:
        run_3dpw(MAIN_MODEL, 'test', s)
    for s in VAL_SEQS:
        run_3dpw(MAIN_MODEL, 'validation', s)
    measure_speed(MAIN_MODEL)
    print('Part A finished.')

## Part B — Real-object occluders (Pascal VOC 2012)
Objects are cut out of Pascal VOC images with their segmentation masks (people are excluded), as in Sárándi et al. (2018), and pasted over the person. **Terms:** the VOC images come from Flickr, and their use must respect Flickr's terms of use (http://www.flickr.com/terms.gne); cite Everingham et al. (IJCV 2010). Set `VOC_TERMS_AGREED = True` in Step 1 to run this part.

1. **Per-frame study:** every 15th test frame, fully visible people, one object covering 10–50% of the person box.
2. **Occlusion episodes:** in every test video, each person is covered for 30 frames out of every 150 by one object (30–50% of the box) that moves with the person. All frames are processed, so the temporal refinement can be tested on occluded stretches.

In [ ]:
VOC_URLS = ['http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar',
            'https://datasets.cms.waikato.ac.nz/ufdl/data/pascalvoc2012/VOCtrainval_11-May-2012.tar']
VOC_MD5 = '6cd6e144f989b92b3379bac3b3de84fd'
VOC_ROOT = '/content/VOC/VOCdevkit/VOC2012'
OCCLUDERS = None
if RUN_VOC_STUDY and not VOC_TERMS_AGREED:
    print('Part B skipped: set VOC_TERMS_AGREED = True in Step 1 after agreeing to the terms above.')
elif RUN_VOC_STUDY:
    os.makedirs('/content/VOC', exist_ok=True)
    tar = '/content/VOC/voc2012.tar'
    if not os.path.isdir(os.path.join(VOC_ROOT, 'SegmentationObject')):
        for url in VOC_URLS:
            if os.path.exists(tar):
                break
            print('Downloading Pascal VOC 2012 from', url)
            r = subprocess.run(['wget', '-q', '--show-progress', '--progress=bar:force', '-c', '-O', tar + '.part', url])
            if r.returncode == 0:
                md5 = subprocess.run(['md5sum', tar + '.part'], capture_output=True, text=True).stdout.split()[0]
                if md5 == VOC_MD5:
                    os.rename(tar + '.part', tar)
                else:
                    print('Checksum mismatch, trying the next mirror'); os.remove(tar + '.part')
        subprocess.run(['tar', '-xf', tar, '-C', '/content/VOC'], check=True)
    OCCLUDERS = load_voc_occluders(VOC_ROOT, max_count=300 if QUICK_TEST else None)
    print(len(OCCLUDERS), 'object occluders loaded')

In [ ]:
def run_voc_static(seq_name):
    out = os.path.join(RESULTS_DIR, f'vocstatic_{MAIN_MODEL}_{seq_name}.pkl')
    if os.path.exists(out):
        return
    model = get_model(MAIN_MODEL)
    seq = load_sequence(SEQ_FILES[seq_name])
    K = np.asarray(seq['cam_intrinsics'], dtype=np.float64); K_tf = tf.constant(K, tf.float32)
    recs = []
    for i in tqdm(range(0, len(seq['cam_poses']), 30 if QUICK_TEST else ROBUST_STRIDE), desc=f'VOC per-frame | {seq_name}'):
        actors = [a for a in range(len(seq['jointPositions'])) if frame_is_valid(seq, a, i)]
        if not actors:
            continue
        img = tf.image.decode_jpeg(tf.io.read_file(os.path.join(IMG_DIRS[seq_name], f'image_{i:05d}.jpg'))).numpy()
        H, W = img.shape[:2]
        for a in actors:
            g3 = to_camera(np.asarray(seq['jointPositions'][a]).reshape(-1, 24, 3)[i], np.asarray(seq['cam_poses'][i]), INVERT_CAM_POSES) * 1000.0
            uv = project(g3, K)
            if (g3[:, 2] <= 0).any() or not ((uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)).all():
                continue
            box = clip_box(gt_box(uv, W, H), W, H)
            for lev in OCC_LEVELS:
                rng = np.random.default_rng([SEQ_INDEX[seq_name], i, a, int(round(lev * 100)), 7])
                occ = OCCLUDERS[int(rng.integers(len(OCCLUDERS)))]
                im2, mask = paste_occluder(img, occ, box, lev, rng.uniform(0.2, 0.8, 2))
                x, y, w, h = [int(round(v)) for v in box]
                pred = model.estimate_poses(tf.constant(im2), tf.constant([box], tf.float32), intrinsic_matrix=K_tf,
                                            skeleton='smpl_24', num_aug=NUM_AUG, average_aug=False)
                pa = pred['poses3d'].numpy()[0]
                m, sg = rootrel_mean_and_sigma(pa.astype(np.float64))
                recs.append({'kind': 'voc', 'level': lev, 'frame': i, 'actor': a, 'cls': occ['cls'],
                             'cover': float(mask[max(y, 0):y + h, max(x, 0):x + w].mean()),
                             'pred': m.astype(np.float32), 'sigma': sg.astype(np.float32),
                             'gt': (g3 - g3[0]).astype(np.float32), 'affected': joints_on_mask(uv, mask)})
    save_pkl(recs, out)

def run_voc_episodes(seq_name):
    out = os.path.join(RESULTS_DIR, f'vocepisodes_{MAIN_MODEL}_{seq_name}.pkl')
    if os.path.exists(out):
        return
    model = get_model(MAIN_MODEL)
    seq = load_sequence(SEQ_FILES[seq_name])
    K = np.asarray(seq['cam_intrinsics'], dtype=np.float64); K_tf = tf.constant(K, tf.float32)
    n, n_act = len(seq['cam_poses']), len(seq['jointPositions'])
    sched = {}
    for a in range(n_act):
        rng = np.random.default_rng([SEQ_INDEX[seq_name], a, 99])
        start = int(rng.integers(0, EPISODE_EVERY - EPISODE_LEN))
        sched[a] = [{'start': s0, 'occ': int(rng.integers(len(OCCLUDERS))), 'cover': float(rng.uniform(*EPISODE_COVER)),
                     'cuv': rng.uniform(0.25, 0.75, 2)} for s0 in range(start, n, EPISODE_EVERY)]
    items = []
    for i in tqdm(range(min(n, 150) if QUICK_TEST else n), desc=f'VOC episodes | {seq_name}'):
        img = None
        boxes, info = [], []
        for a in range(n_act):
            if not frame_is_valid(seq, a, i):
                continue
            g3 = to_camera(np.asarray(seq['jointPositions'][a]).reshape(-1, 24, 3)[i], np.asarray(seq['cam_poses'][i]), INVERT_CAM_POSES) * 1000.0
            if (g3[:, 2] <= 0).any():
                continue
            if img is None:
                img = tf.image.decode_jpeg(tf.io.read_file(os.path.join(IMG_DIRS[seq_name], f'image_{i:05d}.jpg'))).numpy()
                H, W = img.shape[:2]; mask = np.zeros((H, W), bool); im2 = img
            uv = project(g3, K)
            box = clip_box(gt_box(uv, W, H), W, H)
            if box[2] < 8 or box[3] < 8:
                continue
            ep = next((e for e in sched[a] if e['start'] <= i < e['start'] + EPISODE_LEN), None)
            if ep is not None:
                im2, mask = paste_occluder(im2, OCCLUDERS[ep['occ']], box, ep['cover'], ep['cuv'], mask)
            boxes.append(box); info.append((a, g3, uv, ep is not None))
        if not boxes:
            continue
        pred = model.estimate_poses(tf.constant(im2), tf.constant(boxes, tf.float32), intrinsic_matrix=K_tf,
                                    skeleton='smpl_24', num_aug=NUM_AUG, average_aug=False)
        p3 = pred['poses3d'].numpy()
        for k, (a, g3, uv, in_ep) in enumerate(info):
            items.append({'frame': i, 'actor': a, 'pred_aug': p3[k].astype(np.float32), 'gt3d': g3.astype(np.float32),
                          'episode': in_ep, 'affected': joints_on_mask(uv, mask)})
    save_pkl({'seq': seq_name, 'fps': 30.0, 'items': items}, out)

if RUN_VOC_STUDY and OCCLUDERS:
    for s in TEST_SEQS:
        run_voc_static(s)
    for s in TEST_SEQS:
        run_voc_episodes(s)
    print('Part B finished.')

## Part C — MuPoTS-3D (second dataset)
First the small official package is downloaded and its text files (including the terms of use) are printed below. **Read them**, then set `MUPOTS_TERMS_AGREED = True` in Step 1 and run this cell again to download the test set (into Colab only).

In [ ]:
MUPOTS_PKG = 'https://vcai.mpi-inf.mpg.de/projects/SingleShotMultiPerson/content/mupots-3d-eval.zip'
MUPOTS_DIR = '/content/mupots'
MUPOTS_ROOT = None
if RUN_MUPOTS:
    os.makedirs(MUPOTS_DIR, exist_ok=True)
    pkg = os.path.join(MUPOTS_DIR, 'mupots-3d-eval.zip')
    if not os.path.exists(pkg):
        urllib.request.urlretrieve(MUPOTS_PKG, pkg)
    with zipfile.ZipFile(pkg) as z:
        z.extractall(os.path.join(MUPOTS_DIR, 'pkg'))
    texts = {}
    for p in sorted(glob.glob(os.path.join(MUPOTS_DIR, 'pkg', '**', '*'), recursive=True)):
        if os.path.isfile(p) and os.path.getsize(p) < 200000 and not p.endswith(('.mat', '.jpg', '.png', '.zip')):
            try:
                texts[p] = open(p, encoding='utf-8', errors='replace').read()
            except Exception:
                pass
    for p, t in texts.items():
        print('=' * 20, os.path.relpath(p, MUPOTS_DIR), '=' * 20)
        print(t[:4000])
    urls = sorted({u for t in texts.values() for u in re.findall(r'https?://[^\s"\'<>]*MultiPersonTestSet[^\s"\'<>]*\.zip', t)})
    urls += ['https://vcai.mpi-inf.mpg.de/projects/SingleShotMultiPerson/MultiPersonTestSet.zip',
             'http://gvv.mpi-inf.mpg.de/projects/SingleShotMultiPerson/MultiPersonTestSet.zip']
    print('Data URLs to try:', urls)
    found = glob.glob(os.path.join(MUPOTS_DIR, '**', 'TS1', 'annot.mat'), recursive=True)
    if found:
        MUPOTS_ROOT = os.path.dirname(os.path.dirname(found[0]))
    elif not MUPOTS_TERMS_AGREED:
        print('\nPart C paused: read the terms above, then set MUPOTS_TERMS_AGREED = True in Step 1 and run this cell again.')
    else:
        dz = os.path.join(MUPOTS_DIR, 'MultiPersonTestSet.zip')
        for url in urls:
            if os.path.exists(dz):
                break
            print('Downloading', url)
            r = subprocess.run(['wget', '-q', '--show-progress', '--progress=bar:force', '-c', '-O', dz + '.part', url])
            if r.returncode == 0 and zipfile.is_zipfile(dz + '.part'):
                os.rename(dz + '.part', dz)
            elif os.path.exists(dz + '.part'):
                os.remove(dz + '.part')
        with zipfile.ZipFile(dz) as z:
            z.extractall(MUPOTS_DIR)
        MUPOTS_ROOT = os.path.dirname(os.path.dirname(glob.glob(os.path.join(MUPOTS_DIR, '**', 'TS1', 'annot.mat'), recursive=True)[0]))
    print('MuPoTS-3D folder:', MUPOTS_ROOT)

In [ ]:
from scipy.io import loadmat

def _unwrap(v):
    v = np.asarray(v)
    while v.dtype == object and v.size == 1:
        v = np.asarray(v.reshape(-1)[0])
    return v

def load_mupots_seq(k):
    d = os.path.join(MUPOTS_ROOT, f'TS{k}')
    A = loadmat(os.path.join(d, 'annot.mat'))['annotations']
    O = loadmat(os.path.join(d, 'occlusion.mat'))['occlusion_labels']
    nF, nP = A.shape
    a2 = np.zeros((nF, nP, 17, 2)); a3 = np.zeros((nF, nP, 17, 3)); val = np.zeros((nF, nP), bool); occ = np.zeros((nF, nP, 17), bool)
    for i in range(nF):
        for j in range(nP):
            e = A[i, j]
            x2 = _unwrap(e['annot2']).astype(np.float64); x3 = _unwrap(e['annot3']).astype(np.float64)
            a2[i, j] = x2.T if x2.shape[0] == 2 else x2
            a3[i, j] = x3.T if x3.shape[0] == 3 else x3
            val[i, j] = bool(_unwrap(e['isValidFrame']).reshape(-1)[0])
            occ[i, j] = _unwrap(O[i, j]).reshape(-1)[:17].astype(bool)
    return a2, a3, val, occ

def fit_intrinsics(a2, a3, val):
    m = val[..., None] & (a3[..., 2] > 0)
    X, Y, Z = a3[..., 0][m], a3[..., 1][m], a3[..., 2][m]; u, v = a2[..., 0][m], a2[..., 1][m]
    fx, cx = np.linalg.lstsq(np.stack([X / Z, np.ones_like(Z)], 1), u, rcond=None)[0]
    fy, cy = np.linalg.lstsq(np.stack([Y / Z, np.ones_like(Z)], 1), v, rcond=None)[0]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1.0]])
    err = np.median(np.hypot(fx * X / Z + cx - u, fy * Y / Z + cy - v))
    return K, float(err)

MUPOTS_PERM = None
def run_mupots(model_name, k):
    global MUPOTS_PERM
    out = os.path.join(RESULTS_DIR, f'infmupots_{model_name}_TS{k}.pkl')
    if os.path.exists(out):
        return
    model = get_model(model_name)
    if MUPOTS_PERM is None:
        names = [n.decode().split('_')[0] for n in model.per_skeleton_joint_names['mpi_inf_3dhp_17'].numpy()]
        MUPOTS_PERM = [names.index(n) for n in MPI17_NAMES]
        print('Model 3DHP joints:', names)
    a2, a3, val, occ = load_mupots_seq(k)
    K, rep = fit_intrinsics(a2, a3, val)
    print(f'TS{k}: {a2.shape[0]} frames, {a2.shape[1]} people, intrinsics fitted (median reprojection {rep:.2f} px), occluded-label share {occ[val].mean():.3f}')
    K_tf = tf.constant(K, tf.float32)
    recs = []
    nF = a2.shape[0]
    for i in tqdm(range(min(nF, 100) if QUICK_TEST else nF), desc=f'MuPoTS TS{k}'):
        image = tf.image.decode_jpeg(tf.io.read_file(os.path.join(MUPOTS_ROOT, f'TS{k}', f'img_{i:06d}.jpg')))
        H, W = int(image.shape[0]), int(image.shape[1])
        gts = []
        for j in range(a2.shape[1]):
            if val[i, j]:
                uv = a2[i, j]
                inside = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
                gts.append({'actor': j, 'gt3d': a3[i, j].astype(np.float32), 'gt2d': uv.astype(np.float32), 'inside': inside, 'occl': occ[i, j]})
        t0 = time.time()
        pred = model.detect_poses(image, intrinsic_matrix=K_tf, skeleton='mpi_inf_3dhp_17', num_aug=NUM_AUG, average_aug=False)
        p3, p2, bx = pred['poses3d'].numpy(), pred['poses2d'].numpy(), pred['boxes'].numpy()
        dt = time.time() - t0
        if p3.ndim == 3:
            p3, p2 = p3[:, None], p2[:, None]
        recs.append({'frame': i, 'boxes': bx.astype(np.float32), 'p3': p3[:, :, MUPOTS_PERM].astype(np.float32),
                     'p2': p2.mean(1)[:, MUPOTS_PERM].astype(np.float32), 't': dt, 'gts': gts})
    save_pkl({'seq': f'TS{k}', 'split': 'mupots', 'model': model_name, 'fps': 30.0 if k <= 5 else 60.0, 'K': K, 'frames': recs}, out)

if RUN_MUPOTS and MUPOTS_ROOT:
    MUPOTS_SEQS = sorted(int(m.group(1)) for d in glob.glob(os.path.join(MUPOTS_ROOT, 'TS*'))
                         for m in [re.search(r'TS(\d+)$', d)] if m)
    print(len(MUPOTS_SEQS), 'MuPoTS-3D sequences found')
    for k in ([k for k in (1, 6) if k in MUPOTS_SEQS] if QUICK_TEST else MUPOTS_SEQS):
        run_mupots(MAIN_MODEL, k)
    print('Part C finished.')

## Part D — The larger model (EfficientNetV2-M) on the 3DPW test split
The refinement settings chosen for the ResNet-50 model are applied unchanged.

In [ ]:
if RUN_SECOND_MODEL:
    for s in TEST_SEQS:
        run_3dpw(SECOND_MODEL, 'test', s)
    measure_speed(SECOND_MODEL)
    print('Part D finished.')

## Part E1 — Load the saved predictions

In [ ]:
def load_recs(prefix):
    files = sorted(glob.glob(os.path.join(RESULTS_DIR, prefix + '*.pkl')))
    return [load_pkl(p) for p in files]

CTX3 = dict(parents=SMPL_PARENTS, order=topo_order(SMPL_PARENTS), sym=SMPL_SYM, root=0, eval_joints=None,
            pck_thr=50, auc_max=200, max_gap=MAX_GAP)
if RUN_ANALYSIS:
    t0 = time.time()
    val_prep = [prep_record(r, 0) for r in load_recs(f'inf3dpw_{MAIN_MODEL}_validation_')]
    test_prep = [prep_record(r, 0) for r in load_recs(f'inf3dpw_{MAIN_MODEL}_test_')]
    val_tr, test_tr = gt_identity_tracks(val_prep, 0), gt_identity_tracks(test_prep, 0)
    print(len(val_prep), 'validation and', len(test_prep), 'test sequences;', len(val_tr), 'and', len(test_tr), 'person tracks',
          f'({time.time() - t0:.0f} s)')

## Part E2 — SmoothNet (released checkpoint)

In [ ]:
SNET = None
if RUN_ANALYSIS:
    try:
        import torch
        snet_dir = '/content/smoothnet'; os.makedirs(snet_dir, exist_ok=True)
        if not glob.glob(os.path.join(snet_dir, '**', '*.pth*'), recursive=True):
            subprocess.run(['gdown', '--quiet', SMOOTHNET_FILE_ID, '-O', os.path.join(snet_dir, 'download')], check=True)
            dl = os.path.join(snet_dir, 'download')
            if zipfile.is_zipfile(dl) and not any(n.endswith('data.pkl') for n in zipfile.ZipFile(dl).namelist()):
                zipfile.ZipFile(dl).extractall(snet_dir)
            elif subprocess.run(['tar', '-tf', dl], capture_output=True).returncode == 0:
                subprocess.run(['tar', '-xf', dl, '-C', snet_dir], check=True)
            else:
                os.rename(dl, os.path.join(snet_dir, 'checkpoint.pth.tar'))
        cks = sorted(glob.glob(os.path.join(snet_dir, '**', '*.pth*'), recursive=True))
        print('SmoothNet checkpoints found:', [os.path.relpath(c, snet_dir) for c in cks])
        pick = next((c for c in cks if '32' in os.path.basename(c)), cks[0])
        ck = torch.load(pick, map_location='cpu', weights_only=False)
        state = ck.get('state_dict', ck) if isinstance(ck, dict) else ck
        SNET = smoothnet_weights({k: v.numpy() for k, v in state.items() if hasattr(v, 'numpy')})
        SNET['file'] = os.path.relpath(pick, snet_dir)
        print('Using', SNET['file'], '| window', SNET['window'], 'frames |', len(SNET['blocks']), 'residual block(s)')
    except Exception as e:
        print('SmoothNet could not be loaded, it is left out of the comparison:', repr(e))
CTX3['snet'] = SNET

## Part E3 — Choose all settings on the validation split (lowest MPJPE)

In [ ]:
GRIDS = {}
def best_of(name, specs, tracks=None):
    tracks = val_tr if tracks is None else tracks
    res = grid_search(tracks, specs, CTX3)
    GRIDS[name] = [(round(s, 3), sp) for s, sp in res]
    print(f'{name:28s} best validation MPJPE {res[0][0]:.2f} mm  {res[0][1]}')
    return res[0][1]

if RUN_ANALYSIS:
    t0 = time.time()
    QS = [10, 30, 100, 300, 1000, 3000, 10000]
    VAL_BASE = evaluate_tracks(val_tr, {'kind': 'raw'}, CTX3, fast=True)
    print(f'Validation MPJPE without refinement: {VAL_BASE:.2f} mm')
    BEST = {}
    for smooth, tag in ((False, 'online'), (True, 'offline')):
        BEST['fixed_' + tag] = best_of('fixed_' + tag, [{'kind': 'refine', 'cfg': dict(mode='fixed', q=q, rfix=r, smooth=smooth)}
                                                        for q in QS for r in [100, 300, 1000, 3000, 10000]])
        BEST['ug_' + tag] = best_of('ug_' + tag, [{'kind': 'refine', 'cfg': dict(mode='uncertainty', q=q, lam=l, smin=m, smooth=smooth)}
                                                  for q in QS for l in [0.3, 1, 3, 10, 30] for m in [3, 10]])
        hows = ['median', 'mean'] if smooth else ['mean']
        BEST['ugb_' + tag] = best_of('ug_bones_' + tag, [{'kind': 'refine', 'cfg': dict(BEST['ug_' + tag]['cfg'], bones=dict(kappa=k, bsmin=b, s0=5.0, how=h, sym=sy, order=o))}
                                                         for k in [0.5, 1.0] for b in [3, 10, 20] for h in hows for sy in [False, True] for o in ['after', 'before']])
    BEST['bones_only'] = {'kind': 'refine', 'cfg': dict(mode=None, smooth=True, bones=BEST['ugb_offline']['cfg']['bones'])}
    BEST['oneeuro'] = best_of('one_euro', [{'kind': 'smoother', 'method': 'oneeuro', 'params': dict(min_cutoff=c, beta=b)}
                                           for c in [0.05, 0.1, 0.2, 0.5, 1, 2, 5] for b in [0, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]])
    BEST['savgol'] = best_of('savitzky_golay', [{'kind': 'smoother', 'method': 'savgol', 'params': dict(window=w, order=o)}
                                                for w in [5, 7, 9, 13, 17, 21, 31, 41, 61] for o in [1, 2, 3] if o < w])
    BEST['gauss'] = best_of('gaussian', [{'kind': 'smoother', 'method': 'gauss', 'params': dict(sigma=s)} for s in [0.5, 1, 1.5, 2, 3, 4, 6, 8, 12]])
    if SNET is not None:
        BEST['smoothnet'] = {'kind': 'smoother', 'method': 'smoothnet', 'params': {}}
    BEST['baseline'] = {'kind': 'raw'}
    save_json({'val_baseline': VAL_BASE, 'best': BEST, 'grids': GRIDS}, os.path.join(RESULTS_DIR, f'tuning_{MAIN_MODEL}.json'))
    print(f'Tuning took {time.time() - t0:.0f} s')

## Part E4 — Test split: all methods, significance tests, uncertainty analysis

In [ ]:
ORDER = ['baseline', 'oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_online', 'ug_online', 'ugb_online',
         'fixed_offline', 'ug_offline', 'bones_only', 'ugb_offline']
def run_all(tracks, ctx, names=None):
    out = {}
    for n in (names or ORDER):
        if n in BEST:
            t0 = time.time()
            out[n] = evaluate_tracks(tracks, BEST[n], ctx)
            out[n]['ms_per_frame'] = time_method(tracks[:4], BEST[n], ctx)
            print(f'{n:14s} MPJPE {out[n]["MPJPE"]:6.2f}  PA {out[n]["PA-MPJPE"]:6.2f}  PCK {out[n]["PCK"]:5.1f}  Accel {out[n]["Accel"]:6.2f}'
                  f'  coverage {out[n]["coverage_pct"]:5.1f}%  ({time.time() - t0:.0f} s)')
    return out

PAIRS = [('baseline', 'ugb_offline'), ('ug_offline', 'ugb_offline'), ('baseline', 'ug_offline'), ('fixed_offline', 'ug_offline'),
         ('baseline', 'ugb_online'), ('ug_online', 'ugb_online'), ('fixed_online', 'ug_online')]
def sig_tests(res, metric='MPJPE'):
    out = {}
    smoothers = [n for n in ('oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_offline') if n in res]
    best_other = min(smoothers, key=lambda n: res[n]['MPJPE']) if smoothers else None
    pairs = PAIRS + ([(best_other, 'ugb_offline')] if best_other else [])
    for a, b in pairs:
        if a in res and b in res:
            seqs = [s for s in sorted(res[a]['per_seq']) if res[a]['per_seq'][s][metric] is not None
                    and res[b]['per_seq'].get(s, {}).get(metric) is not None]
            xa = [res[a]['per_seq'][s][metric] for s in seqs]; xb = [res[b]['per_seq'][s][metric] for s in seqs]
            w = [res[a]['per_seq'][s]['n' if metric == 'MPJPE' else 'acc_n'] for s in seqs]
            out[f'{b}_vs_{a}_{metric}'] = paired_tests(xa, xb, weights=w)
    return out

if RUN_ANALYSIS:
    RES = {'test_3dpw': run_all(test_tr, CTX3)}
    RES['sig_3dpw'] = {**sig_tests(RES['test_3dpw']), **sig_tests(RES['test_3dpw'], 'Accel')}
    RES['uncertainty_3dpw'] = uncertainty_analysis(test_tr, 0)
    for k, v in RES['sig_3dpw'].items():
        print(f'{k:40s} diff {v["mean_diff"]:+.2f}  95% CI [{v["ci95"][0]:+.2f}, {v["ci95"][1]:+.2f}]  Wilcoxon p = {v["wilcoxon_p"]:.2g}  ({v["n_improved"]}/{v["n_sequences"]} sequences better)')
    save_json(RES, os.path.join(RESULTS_DIR, 'results_main.json'))

## Part E5 — Real tracker instead of the ground-truth identities

In [ ]:
if RUN_ANALYSIS:
    RES['tracker_3dpw'] = {}
    for n in ['baseline', 'fixed_offline', 'ug_offline', 'ugb_offline', 'ug_online', 'ugb_online', 'savgol']:
        if n in BEST:
            r = evaluate_with_tracker(test_prep, BEST[n], CTX3)
            RES['tracker_3dpw'][n] = r
            print(f'{n:14s} MPJPE {r["MPJPE"]:6.2f}  Accel {r["Accel"]:6.2f}  coverage {r["coverage_pct"]:5.1f}%  ID switches {r["id_switches"]}  tracks {r["tracks"]}')
    save_json(RES, os.path.join(RESULTS_DIR, 'results_main.json'))

## Part E6 — How many test-time crops are needed? And how sensitive are the settings?

In [ ]:
SUBSETS = {1: [2], 2: [1, 3], 3: [1, 2, 3], 4: [0, 1, 3, 4], 5: [0, 1, 2, 3, 4]}
if RUN_ANALYSIS:
    RES['crops'] = {}
    recs = load_recs(f'inf3dpw_{MAIN_MODEL}_test_')
    for k, idx in SUBSETS.items():
        tr_k = gt_identity_tracks([prep_record(r, 0, aug_idx=idx) for r in recs], 0)
        row = {}
        for n in ['baseline', 'ug_offline', 'ugb_offline', 'ug_online']:
            r = evaluate_tracks(tr_k, BEST[n], CTX3)
            row[n] = {'MPJPE': r['MPJPE'], 'PA-MPJPE': r['PA-MPJPE'], 'Accel': r['Accel']}
        if k >= 2:
            row['spearman'] = uncertainty_analysis(tr_k, 0)['spearman_sigma_error']
        RES['crops'][k] = row
        print(k, 'crops:', {n: round(v['MPJPE'], 2) for n, v in row.items() if isinstance(v, dict)}, 'spearman', round(row.get('spearman', float('nan')), 3))
    del recs
    # sensitivity of the uncertainty-guided smoother (validation MPJPE)
    G = [(s, sp['cfg']) for s, sp in GRIDS['ug_offline']]
    RES['sensitivity'] = {
        'q': {q: min(s for s, c in G if c['q'] == q) for q in sorted({c['q'] for _, c in G})},
        'lam': {l: min(s for s, c in G if c['lam'] == l) for l in sorted({c['lam'] for _, c in G})},
        'q_at_best_lam': {c['q']: s for s, c in G if c['lam'] == BEST['ug_offline']['cfg']['lam'] and c['smin'] == BEST['ug_offline']['cfg']['smin']},
        'lam_at_best_q': {c['lam']: s for s, c in G if c['q'] == BEST['ug_offline']['cfg']['q'] and c['smin'] == BEST['ug_offline']['cfg']['smin']},
        'val_baseline': VAL_BASE}
    print(RES['sensitivity'])
    save_json(RES, os.path.join(RESULTS_DIR, 'results_main.json'))

## Part E7 — The larger model, MuPoTS-3D, and the occluder studies

In [ ]:
if RUN_ANALYSIS:
    r2 = load_recs(f'inf3dpw_{SECOND_MODEL}_test_')
    if r2:
        tr2 = gt_identity_tracks([prep_record(r, 0) for r in r2], 0)
        print('---', SECOND_MODEL, 'on 3DPW test (settings from', MAIN_MODEL, ')')
        RES['test_3dpw_model2'] = run_all(tr2, CTX3)
        RES['sig_3dpw_model2'] = sig_tests(RES['test_3dpw_model2'])
        RES['uncertainty_3dpw_model2'] = uncertainty_analysis(tr2, 0)
        del r2, tr2
    rm = load_recs(f'infmupots_{MAIN_MODEL}_')
    if rm:
        CTXM = dict(parents=MPI17_PARENTS, order=topo_order(MPI17_PARENTS), sym=MPI17_SYM, root=14, eval_joints=list(range(14)),
                    pck_thr=150, auc_max=150, max_gap=MAX_GAP, snet=SNET)
        mp = [prep_record(r, 14) for r in rm]
        trm = gt_identity_tracks(mp, 14)
        print('--- MuPoTS-3D (settings from 3DPW validation)')
        RES['mupots'] = run_all(trm, CTXM)
        RES['sig_mupots'] = sig_tests(RES['mupots'])
        RES['uncertainty_mupots'] = uncertainty_analysis(trm, 14)
        RES['mupots_tracker'] = {n: evaluate_with_tracker(mp, BEST[n], CTXM) for n in ['baseline', 'ug_offline', 'ugb_offline', 'ugb_online']}
        RES['mupots_info'] = {'sequences': len(rm), 'frames': int(sum(len(r['frames']) for r in rm))}
        del rm, mp, trm
    vs = [x for r in load_recs(f'vocstatic_{MAIN_MODEL}_') for x in r]
    if vs:
        RES['voc_static'] = robust_summary(vs, 0)
        RES['voc_static_cover'] = {str(l): float(np.mean([r['cover'] for r in vs if r['level'] == l])) for l in OCC_LEVELS}
        for k, v in RES['voc_static'].items():
            print(k, {kk: (round(vv, 1) if isinstance(vv, float) else vv) for kk, vv in v.items()})
    ve = load_recs(f'vocepisodes_{MAIN_MODEL}_')
    if ve:
        etr = episode_tracks(ve, 0)
        RES['episodes'] = {}
        for n in ['baseline', 'oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_offline', 'ug_offline', 'ugb_offline', 'fixed_online', 'ug_online', 'ugb_online']:
            if n in BEST:
                RES['episodes'][n] = evaluate_episodes(etr, BEST[n], CTX3)
                e = RES['episodes'][n]
                print(f'{n:14s} occluded frames {e["MPJPE_occluded_frames"]:6.2f}  occluded joints {e["MPJPE_occluded_joints"]:6.2f}  clean frames {e["MPJPE_clean_frames"]:6.2f}')
        RES['sig_episodes'] = {}
        for a, b in [('baseline', 'ugb_offline'), ('ug_offline', 'ugb_offline'), ('fixed_offline', 'ugb_offline'), ('baseline', 'ugb_online')]:
            seqs = sorted(RES['episodes'][a]['per_seq'])
            RES['sig_episodes'][f'{b}_vs_{a}'] = paired_tests([RES['episodes'][a]['per_seq'][s]['MPJPE'] for s in seqs],
                                                              [RES['episodes'][b]['per_seq'][s]['MPJPE'] for s in seqs],
                                                              weights=[RES['episodes'][a]['per_seq'][s]['n'] for s in seqs])
        print(RES['sig_episodes'])
        del ve, etr
    for m in (MAIN_MODEL, SECOND_MODEL):
        p = os.path.join(RESULTS_DIR, f'speed_{m}.json')
        if os.path.exists(p):
            RES[f'speed_{m}'] = json.load(open(p))
    save_json(RES, os.path.join(RESULTS_DIR, 'results_main.json'))
    print('All results saved to', os.path.join(RESULTS_DIR, 'results_main.json'))

## Part E8 — Compact summary (for the paper)

In [ ]:
def _c(x, nd=1):
    if isinstance(x, dict):
        return {str(k): _c(v, nd) for k, v in x.items() if k != 'per_seq'}
    if isinstance(x, (list, tuple)):
        return [_c(v, nd) for v in x]
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else round(float(x), nd)
    return x
if RUN_ANALYSIS:
    KEEP = ['MPJPE', 'PA-MPJPE', 'PCK', 'AUC', 'Accel', 'coverage_pct', 'filled', 'filled_MPJPE', 'MPJPE_outside_image',
            'MPJPE_occluded_joints', 'MPJPE_visible_joints', 'occluded_joint_pct', 'ms_per_frame', 'id_switches', 'tracks', 'evaluated', 'valid']
    COMPACT = {'quick_test': QUICK_TEST, 'best': {k: v for k, v in BEST.items()}}
    for key, val in RES.items():
        if key.startswith(('test_', 'mupots', 'tracker_')) and isinstance(val, dict) and key != 'mupots_info':
            COMPACT[key] = {n: {m: _c(r.get(m), 2 if m == 'ms_per_frame' else 1) for m in KEEP if r.get(m) is not None
                                and not (m in ('valid', 'evaluated') and n != 'baseline')} for n, r in val.items()}
        elif key.startswith('sig_'):
            COMPACT[key] = {n: {'d': round(v['mean_diff'], 2), 'ci': [round(c, 2) for c in v['ci95']], 'p': float(f'{v["wilcoxon_p"]:.2g}'),
                                'better': f'{v["n_improved"]}/{v["n_sequences"]}'} for n, v in val.items()}
        elif key == 'episodes':
            COMPACT[key] = {n: _c({m: r[m] for m in r if m != 'per_seq'}) for n, r in val.items()}
        else:
            COMPACT[key] = _c(val, 3 if key in ('crops', 'sensitivity') else 1)
    txt = json.dumps(COMPACT, separators=(',', ':'))
    with open(os.path.join(RESULTS_DIR, 'results_compact.json'), 'w') as f:
        f.write(txt)
    print(len(txt), 'characters')
    print(txt)
    if QUICK_TEST:
        print('\nQUICK_TEST is on: these numbers are only a check. Do NOT put them in the paper.')

## Part F — Review checks on the saved predictions (CPU)
Shuffled-uncertainty controls, bone-stage ablation, extended λ search, support and tracker identity, gap filling, lag, bone lengths, Holm correction, CPU speed. Results: `results_partF.json`.

In [ ]:
# ===== Review checks (Parts F and G). This one cell runs everything it needs by itself. =====
# Datasets are downloaded into this Colab session only; results go to the Drive folder (results_partF.json, results_partG.json).
import subprocess, os, time, traceback
os.makedirs('/content/3DPW_download', exist_ok=True); os.makedirs('/content/VOC', exist_ok=True); os.makedirs('/content/mupots', exist_ok=True)
_DL = {
 '3dpw': f"cd /content/3DPW_download && for n in sequenceFiles.zip imageFiles.zip; do [ -f $n ] || (wget -q -c -O $n.part {DATA_URL}$n && mv $n.part $n); done",
 'voc': "cd /content/VOC && ([ -f voc2012.tar ] || [ -d VOCdevkit/VOC2012/SegmentationObject ] || (wget -q -c -O voc2012.tar.part http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar && [ \"$(md5sum voc2012.tar.part | cut -d' ' -f1)\" = 6cd6e144f989b92b3379bac3b3de84fd ] && mv voc2012.tar.part voc2012.tar))",
 'mupots': "cd /content/mupots && ([ -f mupots-3d-eval.zip ] || wget -q -O mupots-3d-eval.zip https://vcai.mpi-inf.mpg.de/projects/SingleShotMultiPerson/content/mupots-3d-eval.zip) && ([ -f MultiPersonTestSet.zip ] || (wget -q -c -O MultiPersonTestSet.zip.part https://vcai.mpi-inf.mpg.de/projects/SingleShotMultiPerson/MultiPersonTestSet.zip && mv MultiPersonTestSet.zip.part MultiPersonTestSet.zip))"}
_PROCS = {} if not RUN_REVIEW else {k: subprocess.Popen(['bash', '-c', v], stdout=open(f'/content/dl_{k}.log', 'w'), stderr=subprocess.STDOUT) for k, v in _DL.items()}
if _PROCS:
    print('Downloads started in the background (into Colab only):', list(_PROCS))
def _run(name, src):
    t0 = time.time()
    print(f'\n######## {name} ########', flush=True)
    try:
        exec(src, globals())
    except Exception:
        traceback.print_exc()
        print(f'!!!!!!!! {name} FAILED !!!!!!!!', flush=True)
        return False
    print(f'######## {name} done ({(time.time() - t0) / 60:.1f} min) ########', flush=True)
    return True
if RUN_REVIEW:
    _run('Part F', r'''# ===== Part F: extra checks asked for in the review (CPU is enough) =====
# Needs: Step 1 (QUICK_TEST = False), Step 2, Step 3, Part E1 and Part E2 run first.
# Every section is saved to results_partF.json in the Drive folder; finished sections are skipped on a rerun.
import json, os, time, platform
import numpy as np
from scipy.optimize import linear_sum_assignment

FPATH = os.path.join(RESULTS_DIR, 'results_partF.json')
F = json.load(open(FPATH)) if os.path.exists(FPATH) else {}
def _fsave():
    save_json(F, FPATH)
def _r(x, nd=3):
    if isinstance(x, dict):
        return {str(k): _r(v, nd) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_r(v, nd) for v in x]
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else round(float(x), nd)
    if isinstance(x, np.integer):
        return int(x)
    return x
def _m(r):
    return {k: r.get(k) for k in ('MPJPE', 'PA-MPJPE', 'PCK', 'Accel', 'coverage_pct', 'filled', 'filled_MPJPE') if k in r}
def _say(title):
    print(f'\n=== {title} ===', flush=True)

TUNE = json.load(open(os.path.join(RESULTS_DIR, f'tuning_{MAIN_MODEL}.json')))
BEST = TUNE['best']                      # settings chosen on validation in Part E3
MAIN = json.load(open(os.path.join(RESULTS_DIR, 'results_main.json')))
rm = load_recs(f'infmupots_{MAIN_MODEL}_')
CTXM = dict(parents=MPI17_PARENTS, order=topo_order(MPI17_PARENTS), sym=MPI17_SYM, root=14, eval_joints=list(range(14)),
            pck_thr=150, auc_max=150, max_gap=MAX_GAP, snet=SNET)
mp = [prep_record(r, 14) for r in rm]
trm = gt_identity_tracks(mp, 14)
del rm
print('3DPW test tracks', len(test_tr), '| MuPoTS tracks', len(trm), '| settings:', {k: v.get('cfg', v.get('params')) for k, v in BEST.items() if k.startswith('ugb')})

def with_cfg(name, **kw):
    sp = json.loads(json.dumps(BEST[name]))
    b = kw.pop('bones', None)
    sp['cfg'].update(kw)
    if b is not None:
        sp['cfg']['bones'].update(b)
    return sp

# ---------------------------------------------------------------------------
# F1 (A07, A22) Real versus randomly swapped uncertainty
#   'frames': each joint keeps its own typical sigma, but the values are swapped between frames and people
#   'all'   : every sigma value is swapped with a random one from any joint, frame or person
#   Settings for the swapped versions are chosen again on validation data (with swapped validation sigmas).
# ---------------------------------------------------------------------------
def shuffle_sigma(tracks, how, seed, root):
    rng = np.random.default_rng(seed)
    S = np.concatenate([t['sigma'] for t in tracks])
    keep = [j for j in range(S.shape[1]) if j != root]
    S2 = S.copy()
    if how == 'all':
        S2[:, keep] = rng.permutation(S[:, keep].reshape(-1)).reshape(len(S), len(keep))
    else:
        for j in keep:
            S2[:, j] = rng.permutation(S[:, j])
    out, k = [], 0
    for t in tracks:
        n = len(t['sigma']); t2 = dict(t); t2['sigma'] = S2[k:k + n]; k += n
        out.append(t2)
    return out

def tune_kf(tracks, smooth, ctx):
    specs = [{'kind': 'refine', 'cfg': dict(mode='uncertainty', q=q, lam=l, smin=m, smooth=smooth)}
             for q in [300, 1000, 3000, 10000] for l in [10, 30, 100] for m in [3, 10]]
    return grid_search(tracks, specs, ctx)[0]

def tune_bones(tracks, base, ctx, fixed=None):
    fixed = fixed or {}
    grid = []
    for k in [0.5, 1.0]:
        for b in ([fixed['bsmin']] if 'bsmin' in fixed else [3, 10, 20]):
            sp = json.loads(json.dumps(base)); sp['cfg']['bones'].update(dict(fixed, kappa=k, bsmin=b)); grid.append(sp)
    return grid_search(tracks, grid, ctx)[0]

if 'shuffle' not in F:
    _say('F1 real versus swapped uncertainty'); t0 = time.time()
    out = {'real': {n: _m(MAIN['test_3dpw'][n]) for n in ['ug_offline', 'ugb_offline', 'ug_online', 'ugb_online']}}
    out['real_mupots'] = {n: _m(MAIN['mupots'][n]) for n in ['ug_offline', 'ugb_offline', 'ugb_online'] if 'mupots' in MAIN}
    for how in ['frames', 'all']:
        vs = shuffle_sigma(val_tr, how, 0, 0)
        tuned = {}
        for smooth, tag in ((True, 'offline'), (False, 'online')):
            s, sp = tune_kf(vs, smooth, CTX3)
            tuned['ug_' + tag] = sp
            base = json.loads(json.dumps(sp)); base['cfg']['bones'] = json.loads(json.dumps(BEST['ugb_' + tag]['cfg']['bones']))
            tuned['ugb_' + tag] = tune_bones(vs, base, CTX3)[1]
            print(how, tag, 'validation best', round(s, 2), tuned['ug_' + tag]['cfg'], '| bones', tuned['ugb_' + tag]['cfg']['bones'], flush=True)
        res = {'settings': tuned, 'seeds': {}}
        for seed in ([0, 1, 2] if how == 'frames' else [0]):
            ts = shuffle_sigma(test_tr, how, 100 + seed, 0)
            res['seeds'][seed] = {n: _m(evaluate_tracks(ts, tuned[n], CTX3)) for n in ['ug_offline', 'ugb_offline', 'ug_online', 'ugb_online']}
            res['seeds'][seed]['ugb_offline_same_settings'] = _m(evaluate_tracks(ts, BEST['ugb_offline'], CTX3))
            print(how, 'seed', seed, {n: round(v['MPJPE'], 2) for n, v in res['seeds'][seed].items()}, flush=True)
        # paired per-sequence test: real sigma versus swapped sigma (seed 0), ugb_offline
        ts = shuffle_sigma(test_tr, how, 100, 0)
        a = evaluate_tracks(ts, tuned['ugb_offline'], CTX3)['per_seq']
        b = MAIN['test_3dpw']['ugb_offline']['per_seq']
        seqs = sorted(set(a) & set(b))
        res['test_real_vs_swapped'] = paired_tests([a[s]['MPJPE'] for s in seqs], [b[s]['MPJPE'] for s in seqs], weights=[a[s]['n'] for s in seqs])
        if trm:
            res['mupots'] = {}
            for seed in ([0, 1, 2] if how == 'frames' else [0]):
                tsm = shuffle_sigma(trm, how, 100 + seed, 14)
                res['mupots'][seed] = {n: _m(evaluate_tracks(tsm, tuned[n], CTXM)) for n in ['ug_offline', 'ugb_offline', 'ugb_online']}
            print(how, 'MuPoTS', {s: {n: round(v['MPJPE'], 2) for n, v in d.items()} for s, d in res['mupots'].items()}, flush=True)
        out[how] = res
    F['shuffle'] = _r(out); _fsave(); print(f'({time.time() - t0:.0f} s)')

# ---------------------------------------------------------------------------
# F2 (A07, A22) Bone stage, one part changed at a time (settings of each version chosen on validation)
# ---------------------------------------------------------------------------
if 'ablation' not in F:
    _say('F2 bone-stage ablation'); t0 = time.time()
    out = {}
    for tag in ['offline', 'online']:
        base = BEST['ugb_' + tag]; bb = base['cfg']['bones']
        variants = {'full': None,
                    'plain_lengths': dict(s0=1e6),                      # ordinary (unweighted) bone-length estimate
                    'equal_correction': dict(bsmin=1e6),                # every joint moves the same amount
                    'plain_lengths_equal_correction': dict(s0=1e6, bsmin=1e6),
                    ('separate_left_right' if bb.get('sym', True) else 'shared_left_right'): dict(sym=not bb.get('sym', True)),
                    'other_order': dict(order='before' if bb.get('order', 'after') == 'after' else 'after')}
        out[tag] = {'chosen': bb}
        for v, fx in variants.items():
            if fx is None:
                sp = base
            else:
                b0 = json.loads(json.dumps(base)); b0['cfg']['bones'].update(fx)
                sp = tune_bones(val_tr, b0, CTX3, fixed={k: x for k, x in fx.items()})[1]
            r = evaluate_tracks(test_tr, sp, CTX3)
            out[tag][v] = dict(_m(r), bones=sp['cfg']['bones'])
            print(tag, f'{v:32s} MPJPE {r["MPJPE"]:.3f}  PA {r["PA-MPJPE"]:.3f}  Accel {r["Accel"]:.3f}', flush=True)
        out[tag]['no_bones'] = _m(MAIN['test_3dpw']['ug_' + tag])
    F['ablation'] = _r(out); _fsave(); print(f'({time.time() - t0:.0f} s)')

# ---------------------------------------------------------------------------
# F3 (A18) Noise scale lambda above 30.  With R = lambda * sigma^2 and process noise q, the filter mostly
#    depends on q / lambda, so the search is widened in both.  Validation chooses; test is rerun if it changes.
# ---------------------------------------------------------------------------
if 'lambda' not in F:
    _say('F3 lambda above 30'); t0 = time.time()
    out = {}
    for smooth, tag in ((True, 'offline'), (False, 'online')):
        m = BEST['ug_' + tag]['cfg']['smin']
        specs = [{'kind': 'refine', 'cfg': dict(mode='uncertainty', q=q, lam=l, smin=m, smooth=smooth)}
                 for q in [300, 1000, 3000, 10000, 30000, 100000] for l in [30, 100, 300, 1000]]
        new = grid_search(val_tr, specs, CTX3)
        old = [(s, sp) for s, sp in TUNE['grids']['ug_' + tag]]
        allg = sorted(old + [(s, sp) for s, sp in new], key=lambda x: x[0])
        lam_curve = {}
        for s, sp in allg:
            l = sp['cfg']['lam']; lam_curve[l] = min(lam_curve.get(l, 1e9), s)
        ratio_curve = {}
        for s, sp in allg:
            k = round(sp['cfg']['q'] / sp['cfg']['lam'], 2); ratio_curve[k] = min(ratio_curve.get(k, 1e9), s)
        o = {'old_best': {'val_MPJPE': old[0][0], 'cfg': old[0][1]['cfg']}, 'new_best': {'val_MPJPE': allg[0][0], 'cfg': allg[0][1]['cfg']},
             'best_val_by_lambda': dict(sorted(lam_curve.items())), 'best_val_by_q_over_lambda': dict(sorted(ratio_curve.items())),
             'all_tried': [(round(s, 3), sp['cfg']['q'], sp['cfg']['lam'], sp['cfg']['smin']) for s, sp in allg]}
        if allg[0][0] < old[0][0] - 1e-9:
            nb = allg[0][1]
            o['test_ug_new'] = _m(evaluate_tracks(test_tr, nb, CTX3))
            nbb = json.loads(json.dumps(nb)); nbb['cfg']['bones'] = BEST['ugb_' + tag]['cfg']['bones']
            o['test_ugb_new'] = _m(evaluate_tracks(test_tr, nbb, CTX3))
        print(tag, 'old best', round(old[0][0], 3), old[0][1]['cfg'], '| new best', round(allg[0][0], 3), allg[0][1]['cfg'], flush=True)
        print(tag, 'best validation MPJPE by lambda', {k: round(v, 3) for k, v in sorted(lam_curve.items())}, flush=True)
        out[tag] = o
    F['lambda'] = _r(out); _fsave(); print(f'({time.time() - t0:.0f} s)')

# ---------------------------------------------------------------------------
# F4 (A15, A12) Same people, frames and joints for every method; tracker identity scores
# ---------------------------------------------------------------------------
def support_counts(tracks, ctx):
    J = tracks[0]['pose'].shape[1] if ctx['eval_joints'] is None else len(ctx['eval_joints'])
    nf = sum(len(t['frames']) for t in tracks); trip = 0; gaps = 0
    for t in tracks:
        f = np.asarray(t['frames'])
        run = 1
        for a, b in zip(f[:-1], f[1:]):
            if b == a + 1:
                run += 1
            else:
                trip += max(run - 2, 0); run = 1
                if b - a - 1 <= ctx['max_gap']:
                    gaps += b - a - 1
        trip += max(run - 2, 0)
    return {'person_frames': int(nf), 'joints_per_pose': int(J), 'joint_errors': int(nf * J), 'acceleration_triplets': int(trip),
            'missing_frames_that_can_be_filled': int(gaps), 'annotated_person_frames': int(sum(len(t['gt']) for t in tracks))}

def tracker_items(preps, spec, ctx, iou_thr=0.3, kp_thr=0.5):
    # Same as evaluate_with_tracker, but keeps every evaluated item so that methods can be compared on the same items.
    det_items, fill_items, ids = {}, {}, {}
    n_tracks = 0
    for P in preps:
        fl = [fr['frame'] for fr in P['frames']]
        boxes = {fr['frame']: np.array([d['box'] for d in fr['dets']]).reshape(-1, 4) for fr in P['frames']}
        kps = {fr['frame']: (np.array([d['kp'] for d in fr['dets']]) if fr['dets'] else np.zeros((0, 1, 2))) for fr in P['frames']}
        tid = track_detections(fl, boxes, kps, iou_thr, kp_thr, ctx['max_gap'])
        tr = {}
        for fr in P['frames']:
            for d, t in zip(fr['dets'], tid[fr['frame']]):
                x = tr.setdefault(int(t), {'frames': [], 'pose': [], 'sigma': [], 'kp': []})
                x['frames'].append(fr['frame']); x['pose'].append(d['pose']); x['sigma'].append(d['sigma']); x['kp'].append(d['kp'])
        n_tracks += len(tr)
        cand = {}
        for t, x in tr.items():
            Fr = np.array(x['frames']); KP = np.array(x['kp'])
            est = run_method(Fr, np.array(x['pose']), np.array(x['sigma']), P['fps'], spec, ctx)
            fset = set(Fr.tolist())
            for f, pose in est.items():
                if f in fset:
                    kp = KP[np.searchsorted(Fr, f)]
                else:
                    kp = np.stack([np.stack([np.interp(f, Fr, KP[:, j, c]) for c in range(2)]) for j in range(KP.shape[1])])
                cand.setdefault(f, []).append((t, pose, kp, f in fset))
        for fr in P['frames']:
            f = fr['frame']
            if not fr['gts']:
                continue
            gts2d = [np.asarray(g['gt2d'], np.float64) for g in fr['gts']]
            C = cand.get(f, [])
            det = [c for c in C if c[3]]; fil = [c for c in C if not c[3]]
            a1 = match_detections(np.array([c[2] for c in det]).reshape(-1, gts2d[0].shape[0], 2), gts2d)
            rem = [i for i, a in enumerate(a1) if a is None]
            a2 = match_detections(np.array([c[2] for c in fil]).reshape(-1, gts2d[0].shape[0], 2), [gts2d[i] for i in rem]) if rem and fil else [None] * len(rem)
            final = list(a1)
            for i, a in zip(rem, a2):
                final[i] = ('f', a) if a is not None else None
            for g, a in zip(fr['gts'], final):
                key = (P['seq'], str(g['actor']), int(f))
                g3 = np.asarray(g['gt3d'], np.float64); gt = g3 - g3[ctx['root']]
                if a is None:
                    ids[key] = None; continue
                c = fil[a[1]] if isinstance(a, tuple) else det[a]
                ids[key] = c[0]
                (det_items if c[3] else fill_items)[key] = (c[1], gt)
    return det_items, fill_items, ids, n_tracks

def id_scores(ids):
    # Identity switches (a person's track number changes between two of the person's matched frames) and
    # identity recall: share of annotated person-frames on the one track assigned to that person (one-to-one per sequence).
    seqs = {}
    for (s, a, f), t in ids.items():
        seqs.setdefault(s, []).append((a, f, t))
    sw = 0; idtp = 0; n = 0; matched = 0
    for s, L in seqs.items():
        L.sort(key=lambda x: (x[0], x[1]))
        last = {}
        for a, f, t in L:
            n += 1
            if t is None:
                continue
            matched += 1
            if a in last and last[a] != t:
                sw += 1
            last[a] = t
        acts = sorted({a for a, _, _ in L}); trks = sorted({t for _, _, t in L if t is not None})
        if trks:
            M = np.zeros((len(acts), len(trks)))
            for a, f, t in L:
                if t is not None:
                    M[acts.index(a), trks.index(t)] += 1
            r, c = linear_sum_assignment(-M)
            idtp += M[r, c].sum()
    return {'id_switches': int(sw), 'identity_recall_pct': 100.0 * idtp / max(n, 1), 'matched_pct': 100.0 * matched / max(n, 1),
            'identity_precision_on_annotated_pct': 100.0 * idtp / max(matched, 1),
            'IDF1_on_annotated_people_pct': 100.0 * 2 * idtp / max(n + matched, 1)}

def common_metrics(items_by_method, ctx):
    keys = set.intersection(*[set(d) for d in items_by_method.values()])
    keys = sorted(keys)
    kset = set(keys)
    out = {}
    for n, d in items_by_method.items():
        E, EP, A = [], [], []
        for k in keys:
            e, epa = _align_err(d[k][0], d[k][1], ctx); E.append(e); EP.append(epa)
        for k in keys:
            k1, k2 = (k[0], k[1], k[2] + 1), (k[0], k[1], k[2] + 2)
            if k1 in kset and k2 in kset:
                A.append(accel_error(np.stack([d[x][0] for x in (k, k1, k2)]), np.stack([d[x][1] for x in (k, k1, k2)])))
        r = summarize_r(np.stack(E), np.stack(EP), ctx['pck_thr'], ctx['auc_max'], ctx.get('pck_joints'))
        r['Accel'] = float(np.concatenate(A).mean()) if A else None
        r['items'] = len(keys); r['triplets'] = len(A)
        out[n] = r
    return out

if 'support' not in F:
    _say('F4 same support for every method, tracker identity scores'); t0 = time.time()
    out = {'gt_identity_3dpw': support_counts(test_tr, CTX3), 'gt_identity_mupots': support_counts(trm, CTXM) if trm else None,
           'evaluated_per_method_3dpw': {n: (v.get('evaluated'), v.get('filled')) for n, v in MAIN['test_3dpw'].items()},
           'evaluated_per_method_mupots': {n: (v.get('evaluated'), v.get('filled')) for n, v in MAIN.get('mupots', {}).items()}}
    for name, preps, ctx, methods in [('tracker_3dpw', test_prep, CTX3, ['baseline', 'oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_offline', 'ug_offline', 'ugb_offline', 'ug_online', 'ugb_online']),
                                      ('tracker_mupots', mp, CTXM, ['baseline', 'savgol', 'fixed_offline', 'ug_offline', 'ugb_offline', 'ugb_online'])]:
        if not preps:
            continue
        D, ID, fills = {}, {}, {}
        for n in methods:
            if n not in BEST or (n == 'smoothnet' and SNET is None):
                continue
            det, fil, ids, ntr = tracker_items(preps, BEST[n], ctx)
            D[n] = det; ID[n] = dict(id_scores(ids), tracks=ntr)
            fe = [np.linalg.norm(p - g, axis=1)[ctx['eval_joints']].mean() if ctx['eval_joints'] is not None else np.linalg.norm(p - g, axis=1).mean() for p, g in fil.values()]
            fills[n] = {'filled': len(fe), 'filled_MPJPE': float(np.mean(fe)) if fe else None}
            print(name, n, ID[n], fills[n], flush=True)
        same = common_metrics(D, ctx)
        out[name] = {'same_items': {n: dict(_m(r), items=r['items'], triplets=r['triplets']) for n, r in same.items()},
                     'items_per_method': {n: len(d) for n, d in D.items()}, 'identity': ID, 'filled': fills}
        for n, r in same.items():
            print(f'{name} same items {n:14s} MPJPE {r["MPJPE"]:.3f}  PA {r["PA-MPJPE"]:.3f}  Accel {r["Accel"]:.3f}  items {r["items"]}  triplets {r["triplets"]}', flush=True)
        del D
    F['support'] = _r(out); _fsave(); print(f'({time.time() - t0:.0f} s)')

# ---------------------------------------------------------------------------
# F5 (A09) Gaps that are filled, errors by gap length, start of a track, and delay (lag)
# ---------------------------------------------------------------------------
def gap_study(tracks, spec, ctx):
    J = ctx['eval_joints']
    lens, errs, unfilled = [], [], []
    start_e, later_e = [], []
    for t in tracks:
        est = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], spec, ctx)
        f = np.asarray(t['frames'])
        for a, b in zip(f[:-1], f[1:]):
            g = b - a - 1
            if g <= 0:
                continue
            if g > ctx['max_gap']:
                unfilled.append(g); continue
            for x in range(a + 1, b):
                if x in t['gt'] and x in est:
                    e = np.linalg.norm(est[x] - t['gt'][x], axis=1)
                    lens.append(g); errs.append(e.mean() if J is None else e[J].mean())
        seg_start = f[0]
        for i, x in enumerate(f):
            if i > 0 and x - f[i - 1] - 1 > ctx['max_gap']:
                seg_start = x
            e = np.linalg.norm(est[int(x)] - t['gt'][int(x)], axis=1)
            e = e.mean() if J is None else e[J].mean()
            (start_e if x - seg_start < 5 else later_e).append(e)
    lens, errs = np.array(lens), np.array(errs)
    bins = {'1-2': (1, 2), '3-5': (3, 5), '6-10': (6, 10), '11-15': (11, 15)}
    fps = tracks[0]['fps']
    return {'filled_frames': int(len(errs)), 'longest_filled_gap_frames': int(lens.max()) if len(lens) else 0,
            'longest_filled_gap_s': float(lens.max() / fps) if len(lens) else 0.0,
            'filled_MPJPE_by_gap_length': {k: (float(errs[(lens >= lo) & (lens <= hi)].mean()) if ((lens >= lo) & (lens <= hi)).any() else None,
                                              int(((lens >= lo) & (lens <= hi)).sum())) for k, (lo, hi) in bins.items()},
            'gaps_too_long_to_fill': int(len(unfilled)), 'frames_in_too_long_gaps': int(np.sum(unfilled)) if unfilled else 0,
            'MPJPE_first_5_frames_of_track': float(np.mean(start_e)), 'MPJPE_later_frames': float(np.mean(later_e))}

def lag_study(tracks, spec, ctx, shifts=range(-4, 9)):
    # Delay d (frames) that best lines up the estimated motion with the true motion: compare the estimated
    # velocity at frame t with the true velocity at frame t - d.  A positive d means the output lags behind.
    J = ctx['eval_joints']
    num = {d: 0.0 for d in shifts}; cnt = {d: 0 for d in shifts}
    for t in tracks:
        est = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], spec, ctx)
        fs = sorted(int(x) for x in t['frames'])
        fset = set(fs)
        for d in shifts:
            for x in fs:
                if x - 1 in fset and (x - d) in t['gt'] and (x - d - 1) in t['gt']:
                    ve = est[x] - est[x - 1]; vg = t['gt'][x - d] - t['gt'][x - d - 1]
                    e = np.linalg.norm(ve - vg, axis=1)
                    num[d] += e.mean() if J is None else e[J].mean(); cnt[d] += 1
    curve = {d: num[d] / max(cnt[d], 1) for d in shifts}
    ds = list(shifts); v = np.array([curve[d] for d in ds]); i = int(np.argmin(v))
    frac = 0.0
    if 0 < i < len(ds) - 1:
        den = v[i - 1] - 2 * v[i] + v[i + 1]
        frac = 0.5 * (v[i - 1] - v[i + 1]) / den if den > 0 else 0.0
    lag = ds[i] + frac
    fps = tracks[0]['fps']
    return {'lag_frames': float(lag), 'lag_ms': float(1000 * lag / fps), 'velocity_error_by_shift': curve}

if 'gaps_lag' not in F:
    _say('F5 filled gaps, start of track and delay'); t0 = time.time()
    out = {}
    for name, tracks, ctx in [('3dpw', test_tr, CTX3), ('mupots', trm, CTXM)]:
        if not tracks:
            continue
        out[name] = {'gaps': {n: gap_study(tracks, BEST[n], ctx) for n in ['ugb_offline', 'ugb_online', 'ug_offline', 'ug_online', 'baseline']}}
        print(name, {n: (g['longest_filled_gap_frames'], round(g['longest_filled_gap_s'], 2), g['filled_frames'], g['filled_MPJPE_by_gap_length']) for n, g in out[name]['gaps'].items()}, flush=True)
        out[name]['lag'] = {}
        for n in ['baseline', 'oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_online', 'ug_online', 'ugb_online', 'fixed_offline', 'ug_offline', 'ugb_offline']:
            if n in BEST and not (n == 'smoothnet' and SNET is None):
                out[name]['lag'][n] = lag_study(tracks, BEST[n], ctx)
        print(name, 'lag (frames)', {n: round(v['lag_frames'], 2) for n, v in out[name]['lag'].items()}, flush=True)
    F['gaps_lag'] = _r(out); _fsave(); print(f'({time.time() - t0:.0f} s)')

# ---------------------------------------------------------------------------
# F6 (A10, A11) Bone lengths: pelvis fixed, distance to the targets after the whole pipeline, change over time,
#    and how fast the online length estimate settles (also when the first frames are uncertain)
# ---------------------------------------------------------------------------
def bone_study(tracks, ctx, methods):
    par = ctx['parents']; root = ctx['root']; ch = np.where(par >= 0)[0]
    bcfg = BEST['ugb_offline']['cfg']['bones']; sym = ctx['sym'] if bcfg.get('sym') else None
    out = {}
    gt_std, gt_err = [], []
    for n in methods:
        res_t, std_t, err_t, rootmax = [], [], [], 0.0
        for t in tracks:
            est = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], BEST[n], ctx)
            fs = [int(x) for x in t['frames']]
            X = np.stack([est[f] for f in fs]); G = np.stack([t['gt'][f] for f in fs])
            rootmax = max(rootmax, float(np.abs(X[:, root]).max()))
            L = bone_lengths(X, par)[:, ch]; LG = bone_lengths(G, par)[:, ch]
            target = estimate_bones(t['pose'], t['sigma'], par, sym, bcfg.get('s0', 5.0), bcfg.get('how', 'median'))[ch]
            res_t.append(np.abs(L - target).reshape(-1)); err_t.append(np.abs(L - LG).reshape(-1))
            if len(fs) > 1:
                std_t.append(L.std(0).mean())
                if n == methods[0]:
                    gt_std.append(LG.std(0).mean())
        rr = np.concatenate(res_t); ee = np.concatenate(err_t)
        out[n] = {'pelvis_max_abs_mm': rootmax, 'distance_to_target_mean_mm': float(rr.mean()), 'distance_to_target_p95_mm': float(np.percentile(rr, 95)),
                  'bone_length_error_vs_truth_mm': float(ee.mean()), 'bone_length_std_within_track_mm': float(np.mean(std_t))}
    out['ground_truth_bone_length_std_within_track_mm'] = float(np.mean(gt_std))
    # online start-up: |running estimate after k frames - true length|; all tracks and tracks whose first 10 frames are the most uncertain
    ks = [1, 2, 5, 10, 30, 60, 150]
    first_sig = np.array([t['sigma'][:10].mean() if len(t['sigma']) else np.nan for t in tracks])
    hard = first_sig >= np.nanpercentile(first_sig, 75)
    su = {}
    for label, sel in [('all_tracks', np.ones(len(tracks), bool)), ('uncertain_start_tracks', hard)]:
        acc = {k: [] for k in ks}; off = []
        for t, use in zip(tracks, sel):
            if not use or len(t['frames']) < 2:
                continue
            LG = np.nanmedian(bone_lengths(np.stack(list(t['gt'].values())), par)[:, ch], 0)
            Lon = estimate_bones(t['pose'], t['sigma'], par, sym, bcfg.get('s0', 5.0), online=True)[:, ch]
            for k in ks:
                if len(Lon) >= k:
                    acc[k].append(np.abs(Lon[k - 1] - LG).mean())
            off.append(np.abs(estimate_bones(t['pose'], t['sigma'], par, sym, bcfg.get('s0', 5.0), bcfg.get('how', 'median'))[ch] - LG).mean())
        su[label] = {'online_estimate_error_after_k_frames_mm': {k: (float(np.mean(v)) if v else None, len(v)) for k, v in acc.items()},
                     'offline_estimate_error_mm': float(np.mean(off)) if off else None, 'tracks': int(sel.sum())}
    out['startup'] = su
    return out

if 'bones' not in F:
    _say('F6 bone lengths'); t0 = time.time()
    out = {}
    for name, tracks, ctx in [('3dpw', test_tr, CTX3), ('mupots', trm, CTXM)]:
        if tracks:
            out[name] = bone_study(tracks, ctx, ['baseline', 'ug_offline', 'bones_only', 'ugb_offline', 'ug_online', 'ugb_online'])
            print(name, {n: (round(v['distance_to_target_mean_mm'], 2), round(v['bone_length_std_within_track_mm'], 2), round(v['bone_length_error_vs_truth_mm'], 2), v['pelvis_max_abs_mm'])
                         for n, v in out[name].items() if isinstance(v, dict) and 'pelvis_max_abs_mm' in v}, flush=True)
            print(name, 'start-up', out[name]['startup'], flush=True)
    F['bones'] = _r(out); _fsave(); print(f'({time.time() - t0:.0f} s)')

# ---------------------------------------------------------------------------
# F7 (A19, A24) Per-sequence results, Holm correction of the p-values, unrounded values
# ---------------------------------------------------------------------------
if 'stats' not in F:
    _say('F7 per-sequence table, Holm correction, unrounded values')
    out = {'bootstrap': 'sequences resampled with replacement, 10000 resamples, random seed 0; pose-weighted mean difference; Wilcoxon signed-rank on sequence means'}
    def holm(pvals):
        items = sorted(pvals.items(), key=lambda x: x[1]); m = len(items); adj = {}; run = 0.0
        for i, (k, p) in enumerate(items):
            run = max(run, min(1.0, (m - i) * p)); adj[k] = run
        return adj
    fam = {}
    for key in ['sig_3dpw', 'sig_mupots', 'sig_3dpw_model2', 'sig_episodes']:
        for k, v in MAIN.get(key, {}).items():
            if v.get('wilcoxon_p') is not None and not np.isnan(v['wilcoxon_p']):
                fam[f'{key}:{k}'] = v['wilcoxon_p']
    adj = holm(fam)
    out['tests'] = {k: {'p': fam[k], 'p_holm': adj[k], 'diff': MAIN[k.split(':')[0]][k.split(':')[1]]['mean_diff'],
                        'ci95': MAIN[k.split(':')[0]][k.split(':')[1]]['ci95']} for k in fam}
    for k in sorted(fam, key=lambda x: fam[x]):
        print(f'{k:55s} diff {out["tests"][k]["diff"]:+.3f}  p {fam[k]:.2g}  Holm p {adj[k]:.2g}')
    out['per_sequence'] = {}
    for key in ['test_3dpw', 'mupots']:
        if key in MAIN:
            ms = [n for n in ['baseline', 'savgol', 'gauss', 'oneeuro', 'smoothnet', 'fixed_offline', 'ug_offline', 'ugb_offline', 'ugb_online'] if n in MAIN[key]]
            seqs = sorted(MAIN[key]['baseline']['per_seq'])
            out['per_sequence'][key] = {s: {n: MAIN[key][n]['per_seq'].get(s, {}).get('MPJPE') for n in ms} | {'poses': MAIN[key]['baseline']['per_seq'][s]['n']} for s in seqs}
    out['unrounded'] = {key: {n: {m: MAIN[key][n].get(m) for m in ('MPJPE', 'PA-MPJPE', 'PCK', 'AUC', 'Accel')} for n in MAIN[key]}
                        for key in ['test_3dpw', 'mupots', 'test_3dpw_model2'] if key in MAIN}
    for key, d in out['unrounded'].items():
        print(key, {n: (round(v['MPJPE'], 3), round(v['PA-MPJPE'], 3), round(v['Accel'], 3) if v['Accel'] is not None else None) for n, v in d.items()})
    F['stats'] = _r(out, 4); _fsave()

# ---------------------------------------------------------------------------
# F8 (A26) Refinement speed on this CPU: 3 warm-up runs, then 20 timed runs; same tracks and rules for every method
# ---------------------------------------------------------------------------
if 'speed_cpu' not in F:
    _say('F8 refinement speed on the CPU'); t0 = time.time()
    cpu = ''
    try:
        cpu = [l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')][0]
    except Exception:
        cpu = platform.processor()
    sub = test_tr[:20]
    nfr = sum(len(t['frames']) for t in sub)
    out = {'cpu': cpu, 'cores': os.cpu_count(), 'person_frames_per_run': int(nfr), 'tracks_per_run': len(sub), 'methods': {}}
    for n in ['oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_online', 'ug_online', 'ugb_online', 'fixed_offline', 'ug_offline', 'ugb_offline']:
        if n not in BEST or (n == 'smoothnet' and SNET is None):
            continue
        for _ in range(3):
            time_method(sub, BEST[n], CTX3)
        ts = np.array([time_method(sub, BEST[n], CTX3) for _ in range(20)])
        out['methods'][n] = {'median_ms_per_person_frame': float(np.median(ts)), 'q25': float(np.percentile(ts, 25)), 'q75': float(np.percentile(ts, 75))}
        print(f'{n:14s} {np.median(ts):.4f} ms per person-frame (IQR {np.percentile(ts, 25):.4f}-{np.percentile(ts, 75):.4f})', flush=True)
    F['speed_cpu'] = _r(out, 5); _fsave(); print(f'({time.time() - t0:.0f} s)')

print('\nPart F finished. Saved to', FPATH)
''')
else:
    print('{name} skipped ({flag} = False)')

## Part G — Review checks that need the GPU
MuPoTS-3D with the default camera, model speed, occlusion episodes with more schedules and lengths. Needs Parts B and C (set `RUN_VOC_STUDY`, `RUN_MUPOTS` and the terms flags). Results: `results_partG.json`.

In [ ]:
if RUN_REVIEW:
    _run('Part G', r'''# ===== Part G: GPU checks asked for in the review =====
# Needs: Steps 1-5, Part A cell (for the speed helper), Part B cells, Part C cells, Part E1, E2 (RUN_VOC_STUDY and
# RUN_MUPOTS = True; the saved predictions are found on the Drive, so Parts A-C do not run again).
# G1 (A14) MuPoTS-3D with a camera that is NOT fitted to the test answers (MeTRAbs' default 55-degree field of view)
# G2 (A26) Model speed: warm-up, then 10 repeated timings of 40 frames, for both models
# G3 (A17) Occlusion episodes again with other random seeds and other hiding durations; recovery after the object leaves
import json, os, time, subprocess
import numpy as np

GPATH = os.path.join(RESULTS_DIR, 'results_partG.json')
G = json.load(open(GPATH)) if os.path.exists(GPATH) else {}
def _gsave():
    save_json(G, GPATH)
TUNE = json.load(open(os.path.join(RESULTS_DIR, f'tuning_{MAIN_MODEL}.json')))
BEST = TUNE['best']
MAIN = json.load(open(os.path.join(RESULTS_DIR, 'results_main.json')))
CTXM = dict(parents=MPI17_PARENTS, order=topo_order(MPI17_PARENTS), sym=MPI17_SYM, root=14, eval_joints=list(range(14)),
            pck_thr=150, auc_max=150, max_gap=MAX_GAP, snet=SNET)
def _r(x, nd=3):
    if isinstance(x, dict):
        return {str(k): _r(v, nd) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_r(v, nd) for v in x]
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else round(float(x), nd)
    if isinstance(x, np.integer):
        return int(x)
    return x
def _m(r):
    return {k: r.get(k) for k in ('MPJPE', 'PA-MPJPE', 'PCK', 'AUC', 'Accel', 'coverage_pct') if k in r}
try:
    GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except Exception:
    GPU_NAME = 'unknown'
print('GPU:', GPU_NAME)

# ---------------------------------------------------------------------------
# G1 (A14) MuPoTS-3D without the fitted camera
# ---------------------------------------------------------------------------
def run_mupots_default_cam(model_name, k):
    global MUPOTS_PERM
    out = os.path.join(RESULTS_DIR, f'infmupotsfov_{model_name}_TS{k}.pkl')
    if os.path.exists(out):
        return
    model = get_model(model_name)
    if MUPOTS_PERM is None:
        names = [n.decode().split('_')[0] for n in model.per_skeleton_joint_names['mpi_inf_3dhp_17'].numpy()]
        MUPOTS_PERM = [names.index(n) for n in MPI17_NAMES]
    a2, a3, val, occ = load_mupots_seq(k)
    Kfit, rep = fit_intrinsics(a2, a3, val)
    recs = []
    W = H = None
    for i in tqdm(range(a2.shape[0]), desc=f'MuPoTS TS{k} default camera'):
        image = tf.image.decode_jpeg(tf.io.read_file(os.path.join(MUPOTS_ROOT, f'TS{k}', f'img_{i:06d}.jpg')))
        H, W = int(image.shape[0]), int(image.shape[1])
        gts = []
        for j in range(a2.shape[1]):
            if val[i, j]:
                uv = a2[i, j]
                inside = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
                gts.append({'actor': j, 'gt3d': a3[i, j].astype(np.float32), 'gt2d': uv.astype(np.float32), 'inside': inside, 'occl': occ[i, j]})
        pred = model.detect_poses(image, skeleton='mpi_inf_3dhp_17', num_aug=NUM_AUG, average_aug=False, default_fov_degrees=55)
        p3, p2, bx = pred['poses3d'].numpy(), pred['poses2d'].numpy(), pred['boxes'].numpy()
        if p3.ndim == 3:
            p3, p2 = p3[:, None], p2[:, None]
        recs.append({'frame': i, 'boxes': bx.astype(np.float32), 'p3': p3[:, :, MUPOTS_PERM].astype(np.float32),
                     'p2': p2.mean(1)[:, MUPOTS_PERM].astype(np.float32), 'gts': gts})
    f = max(W, H) / (2 * np.tan(np.radians(55) / 2))
    Kdef = np.array([[f, 0, W / 2], [0, f, H / 2], [0, 0, 1.0]])
    save_pkl({'seq': f'TS{k}', 'split': 'mupots', 'model': model_name, 'fps': 30.0 if k <= 5 else 60.0, 'K': Kdef, 'K_fitted': Kfit,
              'frames': recs}, out)

if 'mupots_default_camera' not in G and MUPOTS_ROOT:
    t0 = time.time()
    seqs = sorted(int(m.group(1)) for d in glob.glob(os.path.join(MUPOTS_ROOT, 'TS*')) for m in [re.search(r'TS(\d+)$', d)] if m)
    for k in seqs:
        run_mupots_default_cam(MAIN_MODEL, k)
    rf = [load_pkl(os.path.join(RESULTS_DIR, f'infmupotsfov_{MAIN_MODEL}_TS{k}.pkl')) for k in seqs]
    cams = {r['seq']: {'fitted_fx': float(r['K_fitted'][0, 0]), 'fitted_fy': float(r['K_fitted'][1, 1]), 'fitted_cx': float(r['K_fitted'][0, 2]),
                       'fitted_cy': float(r['K_fitted'][1, 2]), 'default_f': float(r['K'][0, 0]), 'default_cx': float(r['K'][0, 2]), 'default_cy': float(r['K'][1, 2])} for r in rf}
    trf = gt_identity_tracks([prep_record(r, 14) for r in rf], 14)
    del rf
    out = {'cameras': cams, 'methods': {}, 'fitted_camera_methods': {}}
    for n in ['baseline', 'savgol', 'gauss', 'smoothnet', 'fixed_offline', 'ug_offline', 'ugb_offline', 'ug_online', 'ugb_online']:
        if n in BEST and not (n == 'smoothnet' and SNET is None):
            r = evaluate_tracks(trf, BEST[n], CTXM)
            out['methods'][n] = dict(_m(r), per_seq=r['per_seq'])
            out['fitted_camera_methods'][n] = _m(MAIN['mupots'][n])
            print(f'MuPoTS default camera {n:14s} MPJPE {r["MPJPE"]:.2f}  PA {r["PA-MPJPE"]:.2f}  PCK {r["PCK"]:.1f}  Accel {r["Accel"]:.2f}'
                  f'   | fitted camera MPJPE {MAIN["mupots"][n]["MPJPE"]:.2f}  PCK {MAIN["mupots"][n]["PCK"]:.1f}', flush=True)
    M = out['methods']; sig = {}
    for a, b in [('baseline', 'ugb_offline'), ('baseline', 'ug_offline'), ('ug_offline', 'ugb_offline'), ('baseline', 'ugb_online'), ('fixed_offline', 'ug_offline')]:
        seqs_ = sorted(M[a]['per_seq'])
        sig[f'{b}_vs_{a}'] = paired_tests([M[a]['per_seq'][s]['MPJPE'] for s in seqs_], [M[b]['per_seq'][s]['MPJPE'] for s in seqs_],
                                          weights=[M[a]['per_seq'][s]['n'] for s in seqs_])
    for n in ['baseline', 'ugb_offline']:   # default camera versus fitted camera, same method
        seqs_ = sorted(set(M[n]['per_seq']) & set(MAIN['mupots'][n]['per_seq']))
        sig[f'{n}_default_vs_fitted_camera'] = paired_tests([MAIN['mupots'][n]['per_seq'][s]['MPJPE'] for s in seqs_], [M[n]['per_seq'][s]['MPJPE'] for s in seqs_],
                                                            weights=[M[n]['per_seq'][s]['n'] for s in seqs_])
    out['tests'] = sig
    print({k: (round(v['mean_diff'], 2), [round(c, 2) for c in v['ci95']], f"{v['wilcoxon_p']:.2g}") for k, v in sig.items()})
    out['minutes'] = (time.time() - t0) / 60
    G['mupots_default_camera'] = _r(out); _gsave()

# ---------------------------------------------------------------------------
# G2 (A26) Model speed with warm-up and repeats (the call to detect_poses: detector + 3D network + test-time crops;
#    images are already decoded in memory; 40 frames of the first 3DPW test video; 10 repeats)
# ---------------------------------------------------------------------------
def speed_repeat(model_name, n=40, reps=10):
    model = get_model(model_name)
    s = TEST_SEQS[0]; seq = load_sequence(SEQ_FILES[s])
    K_tf = tf.constant(np.asarray(seq['cam_intrinsics'], np.float32))
    imgs = [tf.image.decode_jpeg(tf.io.read_file(os.path.join(IMG_DIRS[s], f'image_{i:05d}.jpg'))) for i in range(n + 5)]
    res = {'gpu': GPU_NAME, 'sequence': s, 'frames_per_repeat': n, 'repeats': reps}
    for A in (1, 3, 5):
        for im in imgs[:5]:
            model.detect_poses(im, intrinsic_matrix=K_tf, skeleton='smpl_24', num_aug=A)['poses3d'].numpy()
        ts, persons = [], 0
        for _ in range(reps):
            t0 = time.perf_counter()
            for im in imgs[5:]:
                persons += int(model.detect_poses(im, intrinsic_matrix=K_tf, skeleton='smpl_24', num_aug=A)['poses3d'].numpy().shape[0])
            ts.append(1000 * (time.perf_counter() - t0) / n)
        ts = np.array(ts)
        res[f'crops_{A}'] = {'median_ms_per_frame': float(np.median(ts)), 'q25': float(np.percentile(ts, 25)), 'q75': float(np.percentile(ts, 75)),
                             'min': float(ts.min()), 'max': float(ts.max()), 'fps_median': float(1000 / np.median(ts))}
        res['persons_per_frame'] = persons / (n * reps)
        print(model_name, A, 'crops', res[f'crops_{A}'], flush=True)
    return res

if 'speed_gpu' not in G and NEED_3DPW:
    G['speed_gpu'] = _r({m: speed_repeat(m) for m in (MAIN_MODEL, SECOND_MODEL)}); _gsave()

# ---------------------------------------------------------------------------
# G3 (A17) Occlusion episodes with other random seeds and hiding durations.
#    The objects, their size (30-50% of the person box) and the way they are pasted are the same as in Part B;
#    only the random schedule (seed) and the episode length change.  Frames without any object in either the
#    original run or the new schedule are identical to the original run, so their saved predictions are reused.
# ---------------------------------------------------------------------------
def run_voc_episodes_cfg(seq_name, seed, ep_len):
    out = os.path.join(RESULTS_DIR, f'vocepisodes2_s{seed}_L{ep_len}_{MAIN_MODEL}_{seq_name}.pkl')
    if os.path.exists(out):
        return
    old = load_pkl(os.path.join(RESULTS_DIR, f'vocepisodes_{MAIN_MODEL}_{seq_name}.pkl'))
    by_frame = {}
    for it in old['items']:
        by_frame.setdefault(int(it['frame']), []).append(it)
    old_occ = {f for f, L in by_frame.items() if any(bool(it['episode']) for it in L)}
    model = get_model(MAIN_MODEL)
    seq = load_sequence(SEQ_FILES[seq_name])
    K = np.asarray(seq['cam_intrinsics'], dtype=np.float64); K_tf = tf.constant(K, tf.float32)
    n, n_act = len(seq['cam_poses']), len(seq['jointPositions'])
    sched = {}
    for a in range(n_act):
        rng = np.random.default_rng([SEQ_INDEX[seq_name], a, 99, 1000 + seed, ep_len])
        start = int(rng.integers(0, EPISODE_EVERY - ep_len))
        sched[a] = [{'start': s0, 'occ': int(rng.integers(len(OCCLUDERS))), 'cover': float(rng.uniform(*EPISODE_COVER)),
                     'cuv': rng.uniform(0.25, 0.75, 2)} for s0 in range(start, n, EPISODE_EVERY)]
    items, n_run = [], 0
    for i in tqdm(range(n), desc=f'episodes seed {seed} length {ep_len} | {seq_name}'):
        eps = {a: next((e for e in sched[a] if e['start'] <= i < e['start'] + ep_len), None) for a in range(n_act)}
        if i not in old_occ and all(v is None for v in eps.values()):
            for it in by_frame.get(i, []):
                items.append(dict(it, episode=False, affected=np.zeros_like(np.asarray(it['affected']))))
            continue
        img = None
        boxes, info = [], []
        for a in range(n_act):
            if not frame_is_valid(seq, a, i):
                continue
            g3 = to_camera(np.asarray(seq['jointPositions'][a]).reshape(-1, 24, 3)[i], np.asarray(seq['cam_poses'][i]), INVERT_CAM_POSES) * 1000.0
            if (g3[:, 2] <= 0).any():
                continue
            if img is None:
                img = tf.image.decode_jpeg(tf.io.read_file(os.path.join(IMG_DIRS[seq_name], f'image_{i:05d}.jpg'))).numpy()
                H, W = img.shape[:2]; mask = np.zeros((H, W), bool); im2 = img
            uv = project(g3, K)
            box = clip_box(gt_box(uv, W, H), W, H)
            if box[2] < 8 or box[3] < 8:
                continue
            ep = eps[a]
            if ep is not None:
                im2, mask = paste_occluder(im2, OCCLUDERS[ep['occ']], box, ep['cover'], ep['cuv'], mask)
            boxes.append(box); info.append((a, g3, uv, ep is not None))
        if not boxes:
            continue
        n_run += 1
        pred = model.estimate_poses(tf.constant(im2), tf.constant(boxes, tf.float32), intrinsic_matrix=K_tf,
                                    skeleton='smpl_24', num_aug=NUM_AUG, average_aug=False)
        p3 = pred['poses3d'].numpy()
        for k, (a, g3, uv, in_ep) in enumerate(info):
            items.append({'frame': i, 'actor': a, 'pred_aug': p3[k].astype(np.float32), 'gt3d': g3.astype(np.float32),
                          'episode': in_ep, 'affected': joints_on_mask(uv, mask)})
    save_pkl({'seq': seq_name, 'fps': 30.0, 'items': items, 'seed': seed, 'ep_len': ep_len, 'frames_run': n_run}, out)

def recovery_curve(tracks, spec, ctx, K=15):
    # Error in the last 3 occluded frames (k = -2..0) and in the k = 1..K clean frames after the object leaves.
    acc = {k: [] for k in range(-2, K + 1)}
    for tr in tracks:
        est = run_method(tr['frames'], tr['pose'], tr['sigma'], tr['fps'], spec, ctx)
        fs = [int(f) for f in tr['frames']]; fset = set(fs)
        ends = [f for f in fs if tr['episode'][f] and (f + 1) in fset and not tr['episode'][f + 1]]
        for e in ends:
            for k in range(-2, K + 1):
                f = e + k
                if f in fset and (k <= 0 or not tr['episode'][f]):
                    acc[k].append(np.linalg.norm(est[f] - tr['gt'][f], axis=1).mean())
    return {k: (float(np.mean(v)) if v else None) for k, v in acc.items()}

EP_METHODS = ['baseline', 'oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_offline', 'ug_offline', 'ugb_offline', 'fixed_online', 'ug_online', 'ugb_online']
def episode_summary(records):
    etr = episode_tracks(records, 0)
    res = {}
    for n in EP_METHODS:
        if n in BEST and not (n == 'smoothnet' and SNET is None):
            e = evaluate_episodes(etr, BEST[n], CTX3)
            res[n] = {k: v for k, v in e.items() if k != 'per_seq'}; res[n]['per_seq'] = e['per_seq']
    sig = {}
    for a, b in [('baseline', 'ugb_offline'), ('ug_offline', 'ugb_offline'), ('fixed_offline', 'ugb_offline'), ('baseline', 'ugb_online')]:
        seqs = sorted(res[a]['per_seq'])
        sig[f'{b}_vs_{a}'] = paired_tests([res[a]['per_seq'][s]['MPJPE'] for s in seqs], [res[b]['per_seq'][s]['MPJPE'] for s in seqs],
                                          weights=[res[a]['per_seq'][s]['n'] for s in seqs])
    rec = {n: recovery_curve(etr, BEST[n], CTX3) for n in ['baseline', 'savgol', 'fixed_offline', 'ug_offline', 'ugb_offline', 'ugb_online']}
    for n in res:
        res[n].pop('per_seq')
    return {'methods': res, 'tests': sig, 'recovery': rec}

if 'episodes_seed0_L30' not in G:
    G['episodes_seed0_L30'] = _r(episode_summary(load_recs(f'vocepisodes_{MAIN_MODEL}_'))); _gsave()
    print('original episodes', {n: round(v['MPJPE_occluded_frames'], 2) for n, v in G['episodes_seed0_L30']['methods'].items()}, flush=True)
    print('recovery after the object leaves (ugb_offline vs baseline)', G['episodes_seed0_L30']['recovery'], flush=True)

if OCCLUDERS:
    for seed, L in [(1, 30), (2, 30), (0, 15), (0, 60)]:
        key = f'episodes_seed{seed}_L{L}'
        if key in G:
            continue
        t0 = time.time()
        for s in TEST_SEQS:
            run_voc_episodes_cfg(s, seed, L)
        recs = [load_pkl(os.path.join(RESULTS_DIR, f'vocepisodes2_s{seed}_L{L}_{MAIN_MODEL}_{s}.pkl')) for s in TEST_SEQS]
        G[key] = _r(episode_summary(recs)); G[key]['frames_run_on_gpu'] = int(sum(r['frames_run'] for r in recs))
        G[key]['minutes'] = round((time.time() - t0) / 60, 1); _gsave()
        print(key, {n: round(v['MPJPE_occluded_frames'], 2) for n, v in G[key]['methods'].items()}, f"({G[key]['minutes']} min)", flush=True)
        del recs

print('\nPart G finished. Saved to', GPATH)
''')
else:
    print('{name} skipped ({flag} = False)')

## Part H — Example figures (Figures 7–9)

In [ ]:
if RUN_REVIEW:
    _run('Part H', r'''# ===== Part H: matched examples for Figures 6, 8 and 9 (A27, A28, A29) =====
# Each example shows the same frame three ways in the same view and scale: the per-frame MeTRAbs estimate (red),
# the complete offline refinement (green) and the reference joints (white).  Examples are picked automatically:
# the largest improvements, and for Figure 9 also the largest worsening (a failure case).
import os, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIGDIR = os.path.join(RESULTS_DIR, 'figures_review'); os.makedirs(FIGDIR, exist_ok=True)
TUNE = json.load(open(os.path.join(RESULTS_DIR, f'tuning_{MAIN_MODEL}.json'))); BEST = TUNE['best']
HX = {}

def tracks_with_root(rec, root):
    P = prep_record(rec, root)
    roots = {}
    for fr in rec['frames']:
        p3 = np.asarray(fr['p3'], np.float64)
        roots[int(fr['frame'])] = (p3[:, :, root].mean(1) if p3.ndim == 4 else p3[:, root]) if len(p3) else np.zeros((0, 3))
    by = {}
    for fr in P['frames']:
        if not fr['gts']:
            continue
        kp = np.array([d['kp'] for d in fr['dets']]) if fr['dets'] else np.zeros((0, 1, 2))
        assign = match_detections(kp, [np.asarray(g['gt2d'], np.float64) for g in fr['gts']])
        for g, d in zip(fr['gts'], assign):
            t = by.setdefault(g['actor'], {'seq': P['seq'], 'fps': P['fps'], 'actor': g['actor'], 'gt': {}, 'gt_abs': {}, 'gt2d': {},
                                           'inside': {}, 'occl': {}, 'frames': [], 'pose': [], 'sigma': [], 'root_abs': []})
            g3 = np.asarray(g['gt3d'], np.float64); f = int(fr['frame'])
            t['gt'][f] = g3 - g3[root]; t['gt_abs'][f] = g3; t['gt2d'][f] = np.asarray(g['gt2d'], np.float64)
            t['inside'][f] = np.asarray(g['inside'], bool)
            if g.get('occl') is not None:
                t['occl'][f] = np.asarray(g['occl'], bool)
            if d is not None:
                t['frames'].append(f); t['pose'].append(fr['dets'][d]['pose']); t['sigma'].append(fr['dets'][d]['sigma'])
                t['root_abs'].append(roots[f][d])
    out = []
    for t in by.values():
        if len(t['frames']) < 2:
            continue
        J = len(next(iter(t['gt'].values())))
        t['frames'] = np.array(t['frames'], int); t['pose'] = np.array(t['pose']).reshape(-1, J, 3)
        t['sigma'] = np.array(t['sigma']).reshape(-1, J); t['root_abs'] = np.array(t['root_abs']).reshape(-1, 3)
        out.append(t)
    return out

def per_frame_errors(tracks, ctx):
    # (seq, actor, frame) -> dict with both estimates, the reference and joint errors of both methods
    items = {}
    for t in tracks:
        base = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], BEST['baseline'], ctx)
        ref = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], BEST['ugb_offline'], ctx)
        for k, f in enumerate(t['frames']):
            f = int(f)
            items[(t['seq'], t['actor'], f)] = {'base': base[f], 'ref': ref[f], 'gt': t['gt'][f], 'gt_abs': t['gt_abs'][f], 'gt2d': t['gt2d'][f],
                                               'root_abs': t['root_abs'][k], 'inside': t['inside'][f], 'occl': t['occl'].get(f),
                                               'e_base': np.linalg.norm(base[f] - t['gt'][f], axis=1), 'e_ref': np.linalg.norm(ref[f] - t['gt'][f], axis=1)}
    return items

def _img(path):
    try:
        return tf.image.decode_jpeg(tf.io.read_file(path)).numpy()
    except Exception:
        return None

def _canvas(img, x0, y0, x1, y1):
    # crop that may extend beyond the image; outside the image is grey
    x0, y0, x1, y1 = [int(round(v)) for v in (x0, y0, x1, y1)]
    C = np.full((y1 - y0, x1 - x0, 3), 160, np.uint8)
    if img is not None:
        H, W = img.shape[:2]
        sx0, sy0, sx1, sy1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
        if sx1 > sx0 and sy1 > sy0:
            C[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = img[sy0:sy1, sx0:sx1]
    return C

def _skel2d(ax, uv, parents, color, lw=2.0, ls='-'):
    for j, p in enumerate(parents):
        if p >= 0:
            ax.plot([uv[j, 0], uv[p, 0]], [uv[j, 1], uv[p, 1]], color=color, lw=lw, ls=ls, solid_capstyle='round')

def _skel3d_side(ax, X, parents, color, lw=2.0, ls='-'):
    # side view: depth (z, mm) horizontally, height (-y, mm) vertically; root-relative
    for j, p in enumerate(parents):
        if p >= 0:
            ax.plot([X[j, 2], X[p, 2]], [-X[j, 1], -X[p, 1]], color=color, lw=lw, ls=ls, solid_capstyle='round')

def draw_example(ax_img, ax_side, people, img, K, parents, mark=None, title=''):
    # people: list of item dicts (same frame); image overlay of all three skeletons and a side view of the first person
    pts = []
    for it in people:
        for X in (it['base'] + it['root_abs'], it['ref'] + it['root_abs']):
            pts.append(project(X, K))
        pts.append(it['gt2d'])
    P = np.concatenate(pts)
    x0, y0 = P.min(0) - 60; x1, y1 = P.max(0) + 60
    ax_img.imshow(_canvas(img, x0, y0, x1, y1), extent=(x0, x1, y1, y0))
    if img is not None:
        H, W = img.shape[:2]
        ax_img.plot([0, W, W, 0, 0], [0, 0, H, H, 0], color='yellow', lw=1.0, ls=':')
    for it in people:
        _skel2d(ax_img, it['gt2d'], parents, 'white', 3.0)
        _skel2d(ax_img, project(it['base'] + it['root_abs'], K), parents, 'red', 1.8)
        _skel2d(ax_img, project(it['ref'] + it['root_abs'], K), parents, 'lime', 1.8)
        if mark is not None and it.get(mark) is not None:
            m = it[mark]
            ax_img.scatter(it['gt2d'][m, 0], it['gt2d'][m, 1], s=40, facecolors='none', edgecolors='cyan', linewidths=1.5, zorder=5)
    ax_img.set_xlim(x0, x1); ax_img.set_ylim(y1, y0); ax_img.set_xticks([]); ax_img.set_yticks([])
    ax_img.set_title(title, fontsize=8)
    it = people[0]
    _skel3d_side(ax_side, it['gt'], parents, '0.25', 3.0)
    _skel3d_side(ax_side, it['base'], parents, 'red', 1.6)
    _skel3d_side(ax_side, it['ref'], parents, 'green', 1.6)
    ax_side.set_aspect('equal'); ax_side.set_xticks([]); ax_side.set_yticks([])
    ax_side.set_title('side view (depth)', fontsize=8)

def save_fig(examples, fname, title):
    n = len(examples)
    if n == 0:
        print('no examples for', fname, flush=True)
        return None
    fig, axes = plt.subplots(n, 2, figsize=(7.0, 3.1 * n), gridspec_kw={'width_ratios': [2.2, 1]})
    axes = np.atleast_2d(axes)
    for r, ex in enumerate(examples):
        draw_example(axes[r, 0], axes[r, 1], ex['people'], ex['img'], ex['K'], ex['parents'], ex.get('mark'), ex['label'])
    fig.suptitle(title + '\nred: per-frame estimate   green: complete offline refinement   white/grey: reference', fontsize=8)
    fig.tight_layout()
    path = os.path.join(FIGDIR, fname)
    fig.savefig(path, dpi=200); plt.close(fig)
    print('saved', path, flush=True)
    return path

def _summ(people, joints=None):
    eb = np.mean([it['e_base'][joints(it)].mean() if joints else it['e_base'].mean() for it in people])
    er = np.mean([it['e_ref'][joints(it)].mean() if joints else it['e_ref'].mean() for it in people])
    return float(eb), float(er)

# ---------- 3DPW: multi-person (Figure 6) and joints outside the image (Figure 8) ----------
seq_K = {}
items3 = {}
for rec in load_recs(f'inf3dpw_{MAIN_MODEL}_test_'):
    trs = tracks_with_root(rec, 0)
    items3.update(per_frame_errors(trs, CTX3))
    seq_K[rec['seq']] = np.asarray(load_sequence(SEQ_FILES[rec['seq']])['cam_intrinsics'], np.float64) if NEED_3DPW else None
by_frame = {}
for (s, a, f), it in items3.items():
    by_frame.setdefault((s, f), []).append(it)
# Figure 6: frames with two or more annotated people, all detected; score = summed improvement; one frame per sequence
cand = []
for (s, f), L in by_frame.items():
    if len(L) >= 2:
        eb, er = _summ(L)
        cand.append((eb - er, s, f))
cand.sort(reverse=True)
ex6, used = [], set()
for d, s, f in cand:
    if s in used:
        continue
    used.add(s); L = by_frame[(s, f)]; eb, er = _summ(L)
    ex6.append({'people': L, 'img': _img(os.path.join(IMG_DIRS[s], f'image_{f:05d}.jpg')), 'K': seq_K[s], 'parents': SMPL_PARENTS,
                'label': f'{s}, frame {f}: MPJPE {eb:.0f} -> {er:.0f} mm (mean of {len(L)} people)'})
    if len(ex6) == 2:
        break
# Figure 8: people with at least 3 joints outside the image; score = improvement on those joints; one per sequence
cand = []
for (s, a, f), it in items3.items():
    out_j = ~it['inside']
    if out_j.sum() >= 3:
        cand.append((it['e_base'][out_j].mean() - it['e_ref'][out_j].mean(), s, a, f))
cand.sort(reverse=True)
ex8, used = [], set()
for d, s, a, f in cand:
    if s in used:
        continue
    used.add(s); it = items3[(s, a, f)]; it['outside'] = ~it['inside']
    eb, er = _summ([it], lambda x: x['outside'])
    ex8.append({'people': [it], 'img': _img(os.path.join(IMG_DIRS[s], f'image_{f:05d}.jpg')), 'K': seq_K[s], 'parents': SMPL_PARENTS, 'mark': 'outside',
                'label': f'{s}, frame {f}: joints outside the image {eb:.0f} -> {er:.0f} mm'})
    if len(ex8) == 2:
        break
# the same selection rule, worst case (for honesty, reported with the figure)
worst8 = cand[-1] if cand else None
HX['fig8_worst_case'] = {'seq': worst8[1], 'frame': int(worst8[3]), 'change_mm': float(-worst8[0])} if worst8 else None
HX['fig8_share_improved'] = float(np.mean([c[0] > 0 for c in cand])) if cand else None
del items3

# ---------- MuPoTS-3D: natural occlusion (Figure 9), two successes and one failure ----------
items_m, K_m = {}, {}
for rec in load_recs(f'infmupots_{MAIN_MODEL}_'):
    K_m[rec['seq']] = np.asarray(rec['K'], np.float64)
    items_m.update(per_frame_errors(tracks_with_root(rec, 14), CTXM))
cand = []
for (s, a, f), it in items_m.items():
    oc = it['occl']
    if oc is not None and oc[:14].sum() >= 3:
        m = np.zeros(len(it['e_base']), bool); m[:14] = oc[:14]
        cand.append((it['e_base'][m].mean() - it['e_ref'][m].mean(), s, a, f, m))
cand.sort(key=lambda x: -x[0])
ex9, used = [], set()
for d, s, a, f, m in cand:
    if s in used:
        continue
    used.add(s); it = items_m[(s, a, f)]; it['occl_mask'] = m
    eb, er = _summ([it], lambda x: x['occl_mask'])
    ex9.append({'people': [it], 'img': _img(os.path.join(MUPOTS_ROOT, s, f'img_{f:06d}.jpg')), 'K': K_m[s], 'parents': MPI17_PARENTS, 'mark': 'occl_mask',
                'label': f'{s}, frame {f}: occluded joints {eb:.0f} -> {er:.0f} mm (improved)'})
    if len(ex9) == 2:
        break
if cand:
    d, s, a, f, m = cand[-1]
    it = items_m[(s, a, f)]; it['occl_mask'] = m
    eb, er = _summ([it], lambda x: x['occl_mask'])
    ex9.append({'people': [it], 'img': _img(os.path.join(MUPOTS_ROOT, s, f'img_{f:06d}.jpg')), 'K': K_m[s], 'parents': MPI17_PARENTS, 'mark': 'occl_mask',
                'label': f'{s}, frame {f}: occluded joints {eb:.0f} -> {er:.0f} mm (failure: worse)'})
HX['fig9_share_improved'] = float(np.mean([c[0] > 0 for c in cand])) if cand else None
del items_m

HX['files'] = [save_fig(ex6, 'fig6_multiperson.png', 'Multi-person scenes (3DPW test)'),
               save_fig(ex8, 'fig8_truncation.png', 'Joints outside the image (3DPW test); cyan circles: reference joints outside the image'),
               save_fig(ex9, 'fig9_occlusion.png', 'Natural occlusion (MuPoTS-3D); cyan circles: joints labelled occluded')]
HX['labels'] = {'fig6': [e['label'] for e in ex6], 'fig8': [e['label'] for e in ex8], 'fig9': [e['label'] for e in ex9]}
save_json(HX, os.path.join(FIGDIR, 'figure_examples.json'))
print(json.dumps(HX, indent=1), flush=True)
''')
else:
    print('{name} skipped ({flag} = False)')

### Backbone check (ResNet-50 v1 layer names)

In [ ]:
if RUN_REVIEW:
    # Model check for review comment A05: backbone variable names and crop size
    m = get_model(MAIN_MODEL)
    names = [v.name for v in m.crop_model.variables]
    print(len(names), 'variables'); print(names[:6]); print([n for n in names if 'conv1' in n or 'preact' in n or 'stem' in n][:6])
    print(sorted({n.split('/')[0] for n in names})[:20])

### Part H2 — Representative examples (used in the paper)

In [ ]:
if RUN_REVIEW:
    _run('Part H2', r'''# ===== Part H2: representative examples (re-selection). Reuses the drawing helpers defined in Part H. =====
# Rule: keep only frames where the detection clearly belongs to the annotated person (projected per-frame pose within
# 15% of the person's height from the reference 2D joints) and the per-frame error is below 150 mm; then show examples
# near the 90th percentile of improvement (typical clear cases, not the extremes) and, for Figure 9, one example near
# the 5th percentile (a typical failure).
def per_frame_errors2(tracks, ctx, K):
    items = {}
    for t in tracks:
        base = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], BEST['baseline'], ctx)
        ref = run_method(t['frames'], t['pose'], t['sigma'], t['fps'], BEST['ugb_offline'], ctx)
        for k, f in enumerate(t['frames']):
            f = int(f); g2 = t['gt2d'][f]
            h = max(np.ptp(g2[:, 1]), 1.0)
            d2 = np.linalg.norm(project(base[f] + t['root_abs'][k], K) - g2, axis=1).mean() / h
            items[(t['seq'], t['actor'], f)] = {'base': base[f], 'ref': ref[f], 'gt': t['gt'][f], 'gt2d': g2, 'root_abs': t['root_abs'][k],
                                               'inside': t['inside'][f], 'occl': t['occl'].get(f), 'match2d': d2,
                                               'e_base': np.linalg.norm(base[f] - t['gt'][f], axis=1), 'e_ref': np.linalg.norm(ref[f] - t['gt'][f], axis=1)}
    return items

def pick(cand, lo, hi, n, key_seq=1):
    # cand: list of tuples (score, seq, ...) ; take n items with scores between the lo and hi percentiles, one per sequence
    if not cand:
        return []
    sc = np.array([c[0] for c in cand]); a, b = np.percentile(sc, lo), np.percentile(sc, hi)
    mid = 0.5 * (a + b)
    pool = sorted([c for c in cand if a <= c[0] <= b], key=lambda c: abs(c[0] - mid))
    out, used = [], set()
    for c in pool:
        if c[key_seq] in used:
            continue
        used.add(c[key_seq]); out.append(c)
        if len(out) == n:
            break
    return out

HX2 = {}
ok = lambda it: it['match2d'] < 0.15 and it['e_base'].mean() < 150
items3 = {}
for rec in load_recs(f'inf3dpw_{MAIN_MODEL}_test_'):
    K3 = np.asarray(load_sequence(SEQ_FILES[rec['seq']])['cam_intrinsics'], np.float64)
    seq_K[rec['seq']] = K3
    items3.update(per_frame_errors2(tracks_with_root(rec, 0), CTX3, K3))
by_frame = {}
for (s, a, f), it in items3.items():
    by_frame.setdefault((s, f), []).append(it)
cand6 = []
for (s, f), L in by_frame.items():
    if len(L) >= 2 and all(ok(it) and it['inside'].all() for it in L):
        eb, er = _summ(L); cand6.append((eb - er, s, f))
ex6 = []
for d, s, f in pick(cand6, 85, 95, 2):
    L = by_frame[(s, f)]; eb, er = _summ(L)
    ex6.append({'people': L, 'img': _img(os.path.join(IMG_DIRS[s], f'image_{f:05d}.jpg')), 'K': seq_K[s], 'parents': SMPL_PARENTS,
                'label': f'{s}, frame {f}: MPJPE {eb:.0f} -> {er:.0f} mm (mean of {len(L)} people)'})
cand8 = []
for (s, a, f), it in items3.items():
    o = ~it['inside']
    if o.sum() >= 3 and ok(it):
        cand8.append((it['e_base'][o].mean() - it['e_ref'][o].mean(), s, a, f))
ex8 = []
for d, s, a, f in pick(cand8, 85, 95, 2):
    it = items3[(s, a, f)]; it['outside'] = ~it['inside']; eb, er = _summ([it], lambda x: x['outside'])
    ex8.append({'people': [it], 'img': _img(os.path.join(IMG_DIRS[s], f'image_{f:05d}.jpg')), 'K': seq_K[s], 'parents': SMPL_PARENTS, 'mark': 'outside',
                'label': f'{s}, frame {f}: joints outside the image {eb:.0f} -> {er:.0f} mm'})
HX2['fig6_candidates'] = len(cand6); HX2['fig6_median_improvement_mm'] = float(np.median([c[0] for c in cand6])) if cand6 else None
HX2['fig8_candidates'] = len(cand8); HX2['fig8_share_improved'] = float(np.mean([c[0] > 0 for c in cand8])) if cand8 else None
HX2['fig8_median_improvement_mm'] = float(np.median([c[0] for c in cand8])) if cand8 else None
del items3, by_frame

items_m, K_m = {}, {}
for rec in load_recs(f'infmupots_{MAIN_MODEL}_'):
    K_m[rec['seq']] = np.asarray(rec['K'], np.float64)
    items_m.update(per_frame_errors2(tracks_with_root(rec, 14), CTXM, K_m[rec['seq']]))
cand9 = []
for (s, a, f), it in items_m.items():
    oc = it['occl']
    if oc is not None and oc[:14].sum() >= 3 and ok(it):
        m = np.zeros(len(it['e_base']), bool); m[:14] = oc[:14]
        cand9.append((it['e_base'][m].mean() - it['e_ref'][m].mean(), s, a, f, m))
ex9 = []
for d, s, a, f, m in pick(cand9, 85, 95, 2) + pick(cand9, 3, 7, 1):
    it = items_m[(s, a, f)]; it['occl_mask'] = m; eb, er = _summ([it], lambda x: x['occl_mask'])
    tag = 'improved' if er < eb else 'failure: worse'
    ex9.append({'people': [it], 'img': _img(os.path.join(MUPOTS_ROOT, s, f'img_{f:06d}.jpg')), 'K': K_m[s], 'parents': MPI17_PARENTS, 'mark': 'occl_mask',
                'label': f'{s}, frame {f}: occluded joints {eb:.0f} -> {er:.0f} mm ({tag})'})
HX2['fig9_candidates'] = len(cand9); HX2['fig9_share_improved'] = float(np.mean([c[0] > 0 for c in cand9])) if cand9 else None
HX2['fig9_median_improvement_mm'] = float(np.median([c[0] for c in cand9])) if cand9 else None
del items_m
HX2['files'] = [save_fig(ex6, 'fig6_multiperson_v2.png', 'Multi-person scenes (3DPW test)'),
                save_fig(ex8, 'fig8_truncation_v2.png', 'Joints outside the image (3DPW test); cyan circles: reference joints outside the image'),
                save_fig(ex9, 'fig9_occlusion_v2.png', 'Natural occlusion (MuPoTS-3D); cyan circles: joints labelled occluded')]
HX2['labels'] = {'fig6': [e['label'] for e in ex6], 'fig8': [e['label'] for e in ex8], 'fig9': [e['label'] for e in ex9]}
save_json(HX2, os.path.join(FIGDIR, 'figure_examples_v2.json'))
print(json.dumps(HX2, indent=1), flush=True)
''')
else:
    print('{name} skipped ({flag} = False)')

## Part I — HMR 2.0 on 3DPW (an estimator outside the MeTRAbs family)
1. Register at https://smpl.is.tue.mpg.de and download **SMPL for Python, v1.1.0**. Put `basicmodel_neutral_lbs_10_207_0_v1_1_0.pkl` into the Drive results folder (or upload it to `/content`). It is converted inside Colab; nothing is redistributed.
2. Set `RUN_HMR2 = True` in Step 1 and run Steps 1–5, Parts E1 and E2, then the two cells below. The same person boxes as for MeTRAbs are used, with five test-time crops per person.
3. Stage 1 applies the MeTRAbs settings unchanged; stage 2 re-selects them on the HMR 2.0 validation predictions. Results: `results_partI.json`.

In [ ]:
# Part I preparation: install HMR 2.0 (4D-Humans) and download its checkpoint into this Colab session
if RUN_HMR2:
    import subprocess, sys, os, types
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'smplx==0.1.28', 'pytorch-lightning', 'yacs', 'einops', 'timm', 'dill'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'git+https://github.com/shubham-goel/4D-Humans.git'], check=True)
    for _n, _names in [('hmr2.utils.renderer', ['Renderer']), ('hmr2.utils.mesh_renderer', ['MeshRenderer']), ('hmr2.utils.skeleton_renderer', ['SkeletonRenderer'])]:
        _m = types.ModuleType(_n)
        for _a in _names:
            setattr(_m, _a, object)
        sys.modules[_n] = _m
    from hmr2.configs import CACHE_DIR_4DHUMANS
    from hmr2.models import download_models
    download_models(CACHE_DIR_4DHUMANS)
    print('HMR 2.0 files:', os.listdir(CACHE_DIR_4DHUMANS))

In [ ]:
if RUN_HMR2:
    _run('Part I', r'''# ===== Part I (A24): a pose estimator outside the MeTRAbs family - HMR 2.0 (4D-Humans, ICCV 2023) on 3DPW =====
# Same person boxes as the MeTRAbs runs (the detections matched to the annotated people), 5 test-time crops per
# person (original, mirrored, zoomed out, shifted left, darker), sigma = spread of the 5 root-relative SMPL joint sets.
# The refinement settings chosen for MeTRAbs ResNet-50 are applied unchanged; optionally they are re-chosen on the
# HMR 2.0 validation predictions (stage 2).  Needs the SMPL neutral model uploaded to /content (SMPL website, v1.1.0).
import os, sys, glob, json, time, shutil, subprocess, gc
from pathlib import Path
import numpy as np
if subprocess.run([sys.executable, '-c', 'import hmr2, smplx, pytorch_lightning, yacs, einops, timm'], capture_output=True).returncode != 0:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'smplx==0.1.28', 'pytorch-lightning', 'yacs', 'einops', 'timm', 'dill'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'git+https://github.com/shubham-goel/4D-Humans.git'], check=True)
import types
for _n, _names in [('hmr2.utils.renderer', ['Renderer']), ('hmr2.utils.mesh_renderer', ['MeshRenderer']),
                   ('hmr2.utils.skeleton_renderer', ['SkeletonRenderer'])]:     # rendering is not needed; avoids OpenGL
    _m = types.ModuleType(_n)
    for _a in _names:
        setattr(_m, _a, object)
    sys.modules[_n] = _m
import torch, cv2, smplx
from hmr2.configs import CACHE_DIR_4DHUMANS, get_config
from hmr2.models import download_models, DEFAULT_CHECKPOINT, HMR2

IPATH = os.path.join(RESULTS_DIR, 'results_partI.json')
IR = json.load(open(IPATH)) if os.path.exists(IPATH) else {}
TUNE = json.load(open(os.path.join(RESULTS_DIR, f'tuning_{MAIN_MODEL}.json'))); BEST = TUNE['best']

# ---- SMPL neutral model (uploaded by the user to the Colab Files panel or to the Drive results folder) ----
# The SMPL v1.1.0 file stores its shape basis as chumpy objects (Python 2). It is converted once, with a loader that
# only allows the classes this file uses, into a plain numpy pickle with 10 shape components.
smpl_dir = os.path.join(CACHE_DIR_4DHUMANS, 'data', 'smpl'); os.makedirs(smpl_dir, exist_ok=True)
smpl_dst = os.path.join(smpl_dir, 'SMPL_NEUTRAL.pkl')
def _convert_smpl(src, dst):
    import pickle, copyreg, scipy.sparse
    class Ch:
        def __setstate__(self, st):
            self.__dict__.update(st)
    allowed = {('__builtin__', 'object'): object, ('__builtin__', 'set'): set, ('chumpy.ch', 'Ch'): Ch,
               ('copy_reg', '_reconstructor'): copyreg._reconstructor, ('numpy', 'dtype'): np.dtype, ('numpy', 'ndarray'): np.ndarray,
               ('numpy.core.multiarray', '_reconstruct'): np.core.multiarray._reconstruct, ('scipy.sparse.csc', 'csc_matrix'): scipy.sparse.csc_matrix}
    class SafeUnpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if (module, name) in allowed:
                return allowed[(module, name)]
            raise pickle.UnpicklingError(f'blocked {module}.{name}')
    with open(src, 'rb') as f:
        dd = SafeUnpickler(f, encoding='latin1').load()
    out = {}
    for k, v in dd.items():
        if isinstance(v, Ch):
            v = np.asarray(v.__dict__['x'])
        if scipy.sparse.issparse(v):
            v = np.asarray(v.todense())
        out[k] = v
    out['shapedirs'] = np.ascontiguousarray(out['shapedirs'][:, :, :10])
    with open(dst, 'wb') as f:
        pickle.dump(out, f, protocol=4)
if not os.path.exists(smpl_dst):
    cands = []
    for folder in ('/content', RESULTS_DIR, os.path.dirname(RESULTS_DIR), os.path.dirname(os.path.dirname(RESULTS_DIR))):
        if os.path.isdir(folder):
            cands += [os.path.join(folder, f) for f in sorted(os.listdir(folder)) if f.lower().endswith('.pkl') and 'neutral' in f.lower()]
    if not cands:
        raise FileNotFoundError('SMPL neutral model not found: upload basicmodel_neutral_lbs_10_207_0_v1_1_0.pkl to the Colab Files panel or the Drive results folder.')
    _convert_smpl(cands[0], smpl_dst)
    print('SMPL model converted from', cands[0])

# ---- HMR 2.0 checkpoint (downloaded into this Colab session) and model ----
download_models(CACHE_DIR_4DHUMANS)
for m in list(MODELS):
    del MODELS[m]
gc.collect()
hcfg = get_config(str(Path(DEFAULT_CHECKPOINT).parent.parent / 'model_config.yaml'), update_cachedir=True)
if 'BBOX_SHAPE' not in hcfg.MODEL:
    hcfg.defrost(); hcfg.MODEL.BBOX_SHAPE = [192, 256]; hcfg.freeze()
# The released checkpoint stores its config objects next to the weights; PyTorch >= 2.6 refuses these by default,
# so the model is built from the config and only the weights are copied in (official 4D-Humans checkpoint).
HMR = HMR2(hcfg, init_renderer=False)
_ck = torch.load(DEFAULT_CHECKPOINT, map_location='cpu', weights_only=False)
_r = HMR.load_state_dict(_ck['state_dict'], strict=False)
print('HMR 2.0 weights loaded; missing', len(_r.missing_keys), 'unexpected', len(_r.unexpected_keys), flush=True)
del _ck; gc.collect()
HMR = HMR.cuda().eval()
MEAN = 255. * np.array(hcfg.MODEL.IMAGE_MEAN, np.float32); STD = 255. * np.array(hcfg.MODEL.IMAGE_STD, np.float32)
SMPL_SWAP = np.arange(24)
for a, b in SMPL_SYM:
    SMPL_SWAP[a], SMPL_SWAP[b] = b, a
AUGS = [dict(), dict(flip=True), dict(scale=1.15), dict(dx=-0.06), dict(bright=0.75)]

MEAN_T = torch.tensor(MEAN, device='cuda').view(1, 3, 1, 1); STD_T = torch.tensor(STD, device='cuda').view(1, 3, 1, 1)

def warp_patch(img, box, scale=1.0, dx=0.0, size=256):
    # box = [left, top, width, height] (YOLOv4 detection used for MeTRAbs); same cropping as the HMR 2.0 demo
    x, y, w, h = [float(v) for v in box[:4]]
    cx, cy = x + w / 2 + dx * w, y + h / 2
    tw, th = (w, max(w * 256 / 192, h)) if h / max(w, 1e-6) < 256 / 192 else (max(h * 192 / 256, w), h)
    bs = max(tw, th) * scale
    H, W = img.shape[:2]
    r = int(np.ceil(0.75 * bs)) + 2
    x0, y0 = int(np.floor(cx)) - r, int(np.floor(cy)) - r
    reg = np.zeros((2 * r, 2 * r, 3), np.uint8)
    sx0, sy0, sx1, sy1 = max(x0, 0), max(y0, 0), min(x0 + 2 * r, W), min(y0 + 2 * r, H)
    if sx1 > sx0 and sy1 > sy0:
        reg[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = img[sy0:sy1, sx0:sx1]
    f = bs / size / 2.0
    if f > 1.1:
        reg = cv2.GaussianBlur(reg, (0, 0), sigmaX=(f - 1) / 2)
    s = size / bs
    M = np.array([[s, 0, size / 2 - s * (cx - x0)], [0, s, size / 2 - s * (cy - y0)]], np.float32)
    return cv2.warpAffine(reg, M, (size, size), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

def person_patches(img, box):
    # the 5 test-time crops of AUGS: original, mirrored, zoomed out, shifted left, darker (x0.75);
    # mirroring and darkening act on the original crop, exactly as if it were cut again
    b = warp_patch(img, box)
    return [b, np.ascontiguousarray(b[:, ::-1]), warp_patch(img, box, scale=1.15), warp_patch(img, box, dx=-0.06), b], [1, 1, 1, 1, 0.75]

@torch.no_grad()
def hmr_joints(patches, gains):
    # patches: uint8 (B, 256, 256, 3) RGB crops -> SMPL 24 joints (B, 24, 3) in mm, camera-aligned, root not subtracted
    x = torch.from_numpy(np.stack(patches)).cuda().permute(0, 3, 1, 2).float()
    x = (torch.clamp(x * torch.tensor(gains, dtype=torch.float32, device='cuda').view(-1, 1, 1, 1), 0, 255) - MEAN_T) / STD_T
    with torch.autocast('cuda', dtype=torch.float16):
        feats = HMR.backbone(x[:, :, :, 32:-32])
        params, cam, _ = HMR.smpl_head(feats)
    B = x.shape[0]
    out = smplx.SMPLLayer.forward(HMR.smpl, global_orient=params['global_orient'].reshape(B, -1, 3, 3).float(),
                                  body_pose=params['body_pose'].reshape(B, -1, 3, 3).float(), betas=params['betas'].reshape(B, -1).float(),
                                  pose2rot=False)
    return out.joints[:, :24].float().cpu().numpy() * 1000.0

from concurrent.futures import ThreadPoolExecutor
def run_hmr2(split, seq, batch=80, ahead=12):
    out = os.path.join(RESULTS_DIR, f'inf3dpw_hmr2_{split}_{seq}.pkl')
    if os.path.exists(out):
        return
    rec = load_pkl(os.path.join(RESULTS_DIR, f'inf3dpw_{MAIN_MODEL}_{split}_{seq}.pkl'))
    by_frame = {}
    for fi, fr in enumerate(rec['frames']):
        if not fr['gts'] or len(fr['boxes']) == 0:
            continue
        assign = match_detections(np.asarray(fr['p2'], np.float64), [np.asarray(g['gt2d'], np.float64) for g in fr['gts']])
        ds = sorted({a for a in assign if a is not None})
        if ds:
            by_frame[fi] = ds
    order = sorted(by_frame)
    njobs = sum(len(v) for v in by_frame.values())
    def prep(fi):
        # image reading and cropping run in two worker threads while the GPU works on the previous batch
        f = int(rec['frames'][fi]['frame'])
        img = cv2.imread(os.path.join(IMG_DIRS[seq], f'image_{f:05d}.jpg'))[:, :, ::-1]
        return fi, [(d, *person_patches(img, rec['frames'][fi]['boxes'][d])) for d in by_frame[fi]]
    res, P, G, K = {}, [], [], []
    def flush():
        if P:
            J = hmr_joints(P, G)
            for (fi, d, a), j in zip(K, J):
                if AUGS[a].get('flip'):
                    j = j.copy(); j[:, 0] *= -1; j = j[SMPL_SWAP]
                res.setdefault((fi, d), [None] * len(AUGS))[a] = j
            P.clear(); G.clear(); K.clear()
    t0 = time.time()
    with ThreadPoolExecutor(2) as ex, tqdm(total=njobs, desc=f'HMR 2.0 | {split} | {seq}') as bar:
        futs = [ex.submit(prep, fi) for fi in order[:ahead]]
        nxt = len(futs)
        while futs:
            fi, people = futs.pop(0).result()
            if nxt < len(order):
                futs.append(ex.submit(prep, order[nxt])); nxt += 1
            for d, ps, gs in people:
                for a in range(len(AUGS)):
                    P.append(ps[a]); G.append(gs[a]); K.append((fi, d, a))
                bar.update(1)
            if len(P) >= batch:
                flush()
        flush()
    frames = []
    for fi, fr in enumerate(rec['frames']):
        if not fr['gts']:
            continue
        ds = sorted(d for (f2, d) in res if f2 == fi)
        frames.append({'frame': fr['frame'], 'boxes': np.asarray(fr['boxes'])[ds] if ds else np.zeros((0, 5), np.float32),
                       'p3': np.array([np.stack(res[(fi, d)]) for d in ds], np.float32).reshape(len(ds), len(AUGS), 24, 3),
                       'p2': np.asarray(fr['p2'])[ds] if ds else np.zeros((0, 24, 2), np.float32), 'gts': fr['gts']})
    save_pkl({'seq': seq, 'split': split, 'model': 'hmr2', 'fps': 30.0, 'frames': frames,
              'ms_per_crop': 1000 * (time.time() - t0) / max(njobs * len(AUGS), 1)}, out)

def eval_hmr2(best):
    recs = [load_pkl(os.path.join(RESULTS_DIR, f'inf3dpw_hmr2_test_{s}.pkl')) for s in TEST_SEQS]
    tr = gt_identity_tracks([prep_record(r, 0) for r in recs], 0)
    out = {}
    for n in ['baseline', 'oneeuro', 'savgol', 'gauss', 'smoothnet', 'fixed_online', 'ug_online', 'ugb_online', 'fixed_offline', 'ug_offline', 'ugb_offline']:
        if n in best and not (n == 'smoothnet' and SNET is None):
            r = evaluate_tracks(tr, best[n], CTX3); out[n] = r
            print(f'HMR 2.0 {n:14s} MPJPE {r["MPJPE"]:.2f}  PA {r["PA-MPJPE"]:.2f}  PCK {r["PCK"]:.1f}  Accel {r["Accel"]:.2f}', flush=True)
    sig = {}
    for a, b in [('baseline', 'ugb_offline'), ('ug_offline', 'ugb_offline'), ('baseline', 'ug_offline'), ('fixed_offline', 'ug_offline'),
                 ('baseline', 'ugb_online'), ('ug_online', 'ugb_online'), ('fixed_online', 'ug_online')]:
        seqs = sorted(out[a]['per_seq'])
        for met, wk in (('MPJPE', 'n'), ('Accel', 'acc_n')):
            sig[f'{b}_vs_{a}_{met}'] = paired_tests([out[a]['per_seq'][s][met] for s in seqs], [out[b]['per_seq'][s][met] for s in seqs],
                                                    weights=[out[a]['per_seq'][s][wk] for s in seqs])
    unc = uncertainty_analysis(tr, 0)
    print('uncertainty: Spearman', round(unc['spearman_sigma_error'], 3), '| error by sigma quintile', [round(v, 1) for v in unc['error_by_sigma_quintile']], flush=True)
    for k in ('ugb_offline_vs_baseline_MPJPE', 'ug_offline_vs_baseline_MPJPE', 'ug_offline_vs_fixed_offline_MPJPE', 'ugb_offline_vs_ug_offline_MPJPE', 'ugb_online_vs_baseline_MPJPE'):
        v = sig[k]; print(f'{k:36s} {v["mean_diff"]:+.2f} [{v["ci95"][0]:+.2f}, {v["ci95"][1]:+.2f}] p {v["wilcoxon_p"]:.2g} ({v["n_improved"]}/{v["n_sequences"]})', flush=True)
    keep = ('MPJPE', 'PA-MPJPE', 'PCK', 'AUC', 'Accel', 'coverage_pct', 'filled', 'filled_MPJPE', 'MPJPE_outside_image', 'evaluated', 'valid')
    return {'methods': {n: {k: r.get(k) for k in keep} for n, r in out.items()}, 'tests': sig, 'uncertainty': unc,
            'ms_per_crop': float(np.mean([r['ms_per_crop'] for r in recs]))}

# ---- stage 1: test split with the MeTRAbs settings ----
if 'test_metrabs_settings' not in IR:
    for s in TEST_SEQS:
        run_hmr2('test', s)
    IR['test_metrabs_settings'] = eval_hmr2(BEST); save_json(IR, IPATH)

# ---- stage 2: validation split, settings re-chosen for HMR 2.0 (lowest validation MPJPE), then test ----
if 'test_hmr2_settings' not in IR:
    for s in VAL_SEQS:
        run_hmr2('validation', s)
    vtr = gt_identity_tracks([prep_record(load_pkl(os.path.join(RESULTS_DIR, f'inf3dpw_hmr2_validation_{s}.pkl')), 0) for s in VAL_SEQS], 0)
    B2 = json.loads(json.dumps(BEST))
    QS = [100, 300, 1000, 3000, 10000, 30000]
    for smooth, tag in ((True, 'offline'), (False, 'online')):
        B2['fixed_' + tag] = grid_search(vtr, [{'kind': 'refine', 'cfg': dict(mode='fixed', q=q, rfix=r, smooth=smooth)} for q in QS for r in [100, 300, 1000, 3000, 10000]], CTX3)[0][1]
        B2['ug_' + tag] = grid_search(vtr, [{'kind': 'refine', 'cfg': dict(mode='uncertainty', q=q, lam=l, smin=m, smooth=smooth)} for q in QS for l in [3, 10, 30, 100] for m in [3, 10]], CTX3)[0][1]
        bb = []
        for k in [0.5, 1.0]:
            for b in [3, 10, 20]:
                for o in ['after', 'before']:
                    sp = json.loads(json.dumps(B2['ug_' + tag])); sp['cfg']['bones'] = dict(BEST['ugb_' + tag]['cfg']['bones'], kappa=k, bsmin=b, order=o); bb.append(sp)
        B2['ugb_' + tag] = grid_search(vtr, bb, CTX3)[0][1]
    for n in ['oneeuro', 'savgol', 'gauss']:
        grid = {'oneeuro': [{'kind': 'smoother', 'method': 'oneeuro', 'params': dict(min_cutoff=c, beta=b)} for c in [0.1, 0.5, 1, 2, 5] for b in [0, 1e-3, 1e-2]],
                'savgol': [{'kind': 'smoother', 'method': 'savgol', 'params': dict(window=w, order=o)} for w in [5, 9, 13, 21, 31] for o in [1, 2, 3] if o < w],
                'gauss': [{'kind': 'smoother', 'method': 'gauss', 'params': dict(sigma=s)} for s in [0.5, 1, 1.5, 2, 3, 4, 6]]}[n]
        B2[n] = grid_search(vtr, grid, CTX3)[0][1]
    print('settings chosen on HMR 2.0 validation:', {k: B2[k].get('cfg', B2[k].get('params')) for k in B2 if k != 'baseline'}, flush=True)
    IR['hmr2_settings'] = B2
    IR['test_hmr2_settings'] = eval_hmr2(B2); save_json(IR, IPATH)
print('Part I finished. Saved to', IPATH)
''')
else:
    print('{name} skipped ({flag} = False)')

## If something goes wrong
- **Colab disconnected:** open the notebook again and run all cells. Finished parts are read from your Drive and skipped.
- **A download fails:** run the cell again; downloads continue where they stopped.
- **"Cannot connect to a GPU":** Colab's free GPU time is used up for today. Part E (analysis) still runs on the CPU; the GPU parts can continue tomorrow.
- **Any other error:** please open an issue in the GitHub repository with the full error message.